# Automatizzare un'infrastruttura con Python
## L'esempio dei desktop virtuali

**Modulo di 2 ore** all'interno del corso Python di 3 giorni.

In questo notebook impariamo a scrivere un programma Python che **comanda un servizio attraverso la sua API**, cioè attraverso la porta d'ingresso che il servizio riserva ai programmi. Il programma legge un elenco di risorse, lo confronta con una seconda fonte di dati, individua ciò che non torna, agisce e lascia traccia di ciò che ha fatto.

L'esempio che ci accompagna è la gestione di un parco di **desktop virtuali**: computer da ufficio che risiedono in un centro di calcolo e si usano attraverso la rete. Lo schema che impariamo, però, vale per qualunque sistema che si lasci comandare da un programma: un sistema di ticket, un sistema di monitoraggio, un archivio documentale.

**Come si usa questo notebook**

- Le celle si eseguono **in ordine, dall'alto verso il basso**.
- Prima di ogni cella di codice troviamo la spiegazione di ciò che stiamo per fare. Dopo la cella, dove serve, troviamo una guida alla lettura del risultato.
- Ogni termine tecnico viene spiegato la prima volta che compare. In fondo al notebook c'è un **glossario** che li raccoglie tutti.
- I riquadri **«Proviamo noi»** propongono piccoli esercizi. La soluzione è nascosta sotto la scritta «Soluzione» e compare con un clic.

## 0. Assunzioni

Prima di cominciare mettiamo per iscritto le assunzioni su cui questo notebook è costruito. Se una di esse non è vera, è da qui che dobbiamo partire per correggere il materiale.

Alcuni termini che compaiono in questa tabella vengono spiegati più avanti, nel punto in cui servono.

| N. | Assunzione | Che cosa comporta |
|---|---|---|
| A1 | **Platea mista.** Una parte dei partecipanti gestisce desktop virtuali su Google Cloud. Gli altri si occupano di sistemi diversi. | Il taglio è generale. I desktop virtuali sono l'esempio, non l'argomento. |
| A2 | **Conoscenze di partenza.** Conosciamo le basi di Python (variabili, liste, dizionari, cicli, funzioni) e le operazioni elementari di pandas (leggere una tabella, selezionare righe e colonne). | Questi argomenti non vengono rispiegati. Tutto il resto sì. |
| A3 | **Ambiente di lavoro.** Abbiamo Python 3.9 o successivo, con i pacchetti `requests` e `pandas`, e lavoriamo in Jupyter oppure in VS Code. Possiamo scrivere file nella cartella in cui si trova il notebook. | Le sezioni da 1 a 10 non richiedono Internet. |
| A4 | **Nessun sistema reale.** Dall'aula non operiamo sull'infrastruttura reale della banca. | Tutto il laboratorio gira su un **simulatore**, cioè su un programma che imita un sistema reale. Il simulatore è contenuto in questo notebook. |
| A5 | **Dati inventati.** Desktop, matricole, uffici, costi e password del simulatore sono inventati. Non compare alcun dato personale reale. Il simulatore non imita alcun prodotto in commercio. | I numeri servono a esercitarsi, non a fare stime. |
| A6 | **Che cosa intendiamo per «Google VDI».** Assumiamo che i desktop virtuali della banca siano macchine virtuali del servizio Compute Engine di Google Cloud. | È un'ipotesi da verificare con chi li gestisce. Se non è vera, cambia la sezione 11. |
| A7 | **La sezione 11 è dimostrativa.** Il codice per Google Cloud si legge e si commenta in aula. È ricalcato sugli esempi ufficiali di Google, ma non è stato eseguito su un progetto reale. | Va provato in un ambiente di prova prima di qualunque uso. |
| A8 | **Tempi.** I tempi indicati nella mappa qui sotto sono orientativi e sommano a 2 ore. | Gli esercizi segnati come facoltativi si possono saltare. |

### Mappa del modulo

| Parte | Sezioni | A chi interessa | Tempo |
|---|---|---|---|
| Lo schema generale | 1 | Tutti | 20 min |
| Laboratorio con il simulatore | da 2 a 10 | Tutti | 60 min |
| Finestra sul caso reale: Google Cloud | 11 | Dimostrazione del docente. Riguarda da vicino chi gestisce i desktop virtuali | 25 min |
| Lo stesso schema, con il nostro sistema | 12 e 13 | Tutti | 15 min |

Seguono il riepilogo (sezione 14), il glossario e due appendici di consultazione.

**Se il tempo stringe.** Il laboratorio è denso. Senza perdere il filo possiamo saltare gli esercizi 2, 3 e 5 e il paragrafo 10.3: il percorso dei cinque passi resta completo.

---
## 1. Lo schema generale: cinque passi

### 1.1 Che cosa significa «automatizzare un'infrastruttura»

Chiamiamo **infrastruttura** l'insieme delle risorse tecniche che fanno funzionare i servizi informatici: computer, reti, archivi, e i programmi che li governano.

Gestire un'infrastruttura significa ripetere spesso le stesse operazioni: controllare che cosa c'è, verificare che sia in ordine, correggere ciò che non va. Di solito lo facciamo a mano, con il mouse, dentro una **console**, cioè la pagina web o il programma di amministrazione del sistema.

**Automatizzare** significa far eseguire queste operazioni a un programma. I vantaggi sono tre:

- il programma non si stanca e non salta righe, nemmeno quando l'elenco da controllare è lungo;
- il programma esegue ogni volta le stesse operazioni nello stesso modo;
- il programma lascia una traccia scritta di ciò che ha fatto.

### 1.2 Che cos'è una API

Un programma non sa usare il mouse. Per comandare un sistema ha bisogno di una porta d'ingresso pensata apposta per i programmi: la **API**.

**API** è la sigla di *Application Programming Interface*, cioè «interfaccia di programmazione». È l'elenco delle operazioni che un sistema mette a disposizione degli altri programmi, insieme alle regole per richiederle: «dammi l'elenco dei desktop», «spegni questo desktop», e così via.

Possiamo pensare alla API come allo **sportello** di un ufficio. Noi non entriamo negli archivi. Presentiamo allo sportello una richiesta scritta nel modo previsto e riceviamo una risposta. Chi sta dietro lo sportello controlla chi siamo e che cosa ci è permesso chiedere.

Oggi quasi tutte le API viaggiano su **HTTP**, lo stesso protocollo che il browser usa per caricare le pagine web. Un *protocollo* è un insieme di regole che due programmi seguono per capirsi. Il meccanismo è sempre lo stesso:

- il nostro programma invia una **richiesta** a un indirizzo;
- il servizio restituisce una **risposta**.

Le API costruite in questo modo vengono spesso chiamate *API REST*. Per il nostro lavoro basta sapere che il nome indica questo stile: richieste HTTP a indirizzi ben definiti, e risposte in un formato di testo che impareremo a leggere tra poco.

```text
   il nostro programma Python                           il servizio
  +---------------------------+    richiesta HTTP    +-----------------+
  |  1. autenticarsi          | -------------------> |                 |
  |  2. leggere               |                      |       API       |
  |  3. decidere              | <------------------- | (lo sportello)  |
  |  4. agire                 |       risposta       |                 |
  |  5. registrare            |                      +-----------------+
  +---------------------------+
```

### 1.3 I cinque passi

Qualunque programma che comanda un sistema attraverso la sua API segue, in sostanza, questi cinque passi.

| Passo | Che cosa facciamo | Nel nostro esempio |
|---|---|---|
| **1. Autenticarsi** | Diciamo al servizio chi siamo | Ci presentiamo con nome utente e password e riceviamo un lasciapassare temporaneo |
| **2. Leggere** | Chiediamo al servizio come stanno le cose | Scarichiamo l'elenco dei desktop virtuali |
| **3. Decidere** | Confrontiamo ciò che abbiamo letto con ciò che dovrebbe essere | Incrociamo l'elenco con il file delle assegnazioni e cerchiamo le anomalie |
| **4. Agire** | Chiediamo al servizio di cambiare qualcosa | Spegniamo i desktop che nessuno usa |
| **5. Registrare** | Lasciamo traccia di ciò che abbiamo fatto | Salviamo un report e l'elenco delle azioni eseguite |

I passi 2 e 3 sono **senza rischi**: leggere e ragionare non cambia nulla nel sistema. Il passo 4 è quello delicato. Per questo impareremo a eseguirlo prima «per finta», e solo dopo davvero.

### 1.4 L'esempio: i desktop virtuali

Un **desktop virtuale** è un computer da ufficio che non sta sulla scrivania, ma in un centro di calcolo. L'utente lo vede e lo usa attraverso la rete, da un dispositivo qualsiasi. L'insieme delle tecnologie che lo rendono possibile si chiama **VDI**, sigla di *Virtual Desktop Infrastructure*, cioè «infrastruttura di desktop virtuali».

Ogni desktop virtuale è una **macchina virtuale**: un computer realizzato via software, che funziona insieme a molti altri sullo stesso computer fisico.

Per chi lo gestisce, un parco di desktop virtuali è prima di tutto un **inventario**: un elenco di macchine, ciascuna con un nome, uno stato (accesa o spenta), una dimensione e un costo. È un caso semplice da capire anche per chi non se ne occupa, perché somiglia a qualunque altro inventario: utenze, licenze, server, apparati di rete.

### 1.5 Il problema che risolveremo

Immaginiamo che il responsabile dei desktop virtuali ci chieda un controllo periodico, con tre domande.

1. Quali desktop esistono, ma **non risultano assegnati a nessuno**?
2. Quali assegnazioni riguardano **desktop che non esistono**?
3. Quali desktop sono **accesi ma inutilizzati** da più di 30 giorni, e quanto costano?

Alla fine ci chiede di **spegnere** i desktop del punto 3.

Per rispondere abbiamo due fonti:

- il **servizio di gestione dei desktop**, che conosce l'elenco delle macchine e il loro stato;
- un **file delle assegnazioni**, tenuto da un altro ufficio, che dice quale persona ha in carico quale desktop.

Questo lavoro si chiama **riconciliazione**: confrontiamo due elenchi che dovrebbero descrivere la stessa realtà e cerchiamo i punti in cui non coincidono. A mano richiede ore, e la volta successiva va rifatto da capo. Con Python richiede pochi secondi, e possiamo ripeterlo ogni settimana.

---
## 2. Prepariamo l'ambiente

Usiamo due **pacchetti**. Un pacchetto, detto anche *libreria*, è una raccolta di codice già scritto da altri, che aggiungiamo a Python per non ripartire da zero.

- **requests** serve a inviare richieste HTTP e a leggere le risposte. È lo strumento con cui parleremo con il servizio.
- **pandas** serve a lavorare con tabelle di dati. Lo usiamo per trasformare le risposte in tabelle e per incrociarle.

La cella seguente importa i due pacchetti e ne stampa la versione. Importiamo anche quattro **moduli** che fanno già parte di Python e che useremo più avanti. Un modulo è un file di codice pronto all'uso: i pacchetti sono fatti di moduli.

Nei commenti della cella compaiono due termini, JSON e variabili d'ambiente, che spiegheremo nel punto in cui ci serviranno.

In [ ]:
# "import" rende disponibile in questo notebook il codice di un modulo o di un pacchetto.
import json      # lettura e scrittura del formato JSON
import os        # accesso alle variabili d'ambiente del sistema operativo
import sys       # informazioni sull'interprete Python in uso
import time      # funzioni legate al tempo, per esempio "attendi un secondo"

import pandas as pd    # tabelle di dati; "pd" è l'abbreviazione usata da tutti
import requests        # invio di richieste HTTP

print("Python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("requests:", requests.__version__)

**Se la cella si ferma con l'errore `ModuleNotFoundError`**, uno dei due pacchetti non è installato nell'ambiente in uso. Lo installiamo da un terminale, con l'ambiente attivo, in uno di questi due modi:

```text
conda install -c conda-forge --override-channels pandas requests
```

oppure

```text
python -m pip install pandas requests
```

Dopo l'installazione riavviamo il notebook e rieseguiamo la cella.

---
## 3. Accendiamo il simulatore

### 3.1 Che cos'è il simulatore

Un **simulatore** è un programma che imita il comportamento di un sistema reale, in modo da poterci esercitare senza rischi. Il nostro imita un servizio di gestione dei desktop virtuali.

Tecnicamente è un piccolo **servizio web**: un programma che resta in ascolto, riceve richieste HTTP e risponde. Gira sul nostro stesso PC e soltanto lì. Nessuno lo può raggiungere dall'esterno, e il simulatore non contatta nessuno.

Il simulatore contiene:

- **48 desktop virtuali** inventati, con nome, stato, dimensione, costo e data dell'ultimo accesso;
- **due utenze** finte, ciascuna con un proprio **ruolo**, cioè con un proprio insieme di permessi;
- il file **`assegnazioni.csv`**, che scrive nella cartella del notebook al momento dell'avvio.

| Nome utente | Password | Ruolo | Che cosa può fare |
|---|---|---|---|
| `lettore` | `aula-lettura` | lettura | Leggere l'inventario |
| `operatore` | `aula-operativa` | operativo | Leggere l'inventario, accendere e spegnere i desktop |

Queste password sono scritte in chiaro perché sono finte. Nella sezione 5 vedremo come si trattano le password vere.

### 3.2 La cella di servizio

La cella qui sotto contiene il codice del simulatore. È lunga, circa 350 righe, ma **va solo eseguita**: per seguire il laboratorio non è necessario leggerla. Chi è curioso trova una guida alla lettura nell'Appendice A.

Dopo averla eseguita possiamo scorrere direttamente fino al paragrafo 3.3.

In [ ]:
# CELLA DI SERVIZIO - IL SIMULATORE: va solo eseguita, non è necessario leggerla.
# ======================================================================
#  Chi è curioso trova una guida alla lettura nell'Appendice A.
# ======================================================================
import json
import random
import secrets
import socketserver
import threading
import time
from datetime import datetime, timedelta
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urlparse

# ----------------------------------------------------------------------
# 1) I dati inventati
# ----------------------------------------------------------------------

# Utenze finte del simulatore: nome utente -> (password, ruolo).
UTENZE_FINTE = {
    "lettore": ("aula-lettura", "lettura"),
    "operatore": ("aula-operativa", "operativo"),
}

# Per finta, spegnere o accendere un desktop richiede questo tempo (in secondi).
DURATA_OPERAZIONE = 1.0

# Numeri dei desktop che hanno una particolarità: servono a creare le anomalie.
SENZA_ASSEGNAZIONE = {5, 12, 19, 27, 33, 41}             # assenti dal file CSV
ACCESI_E_INUTILIZZATI = {3, 5, 12, 16, 22, 30, 41, 44}   # accesi, ma fermi da tempo
SPENTI = {8, 19, 24, 33, 37}                             # spenti
ASSEGNAZIONI_SENZA_DESKTOP = ["vdi-0052", "vdi-0057"]    # nel CSV, ma inesistenti


def _intero(caso, minimo, massimo):
    """Numero intero 'a caso' tra minimo e massimo, sempre uguale a parità di seme."""
    return minimo + int(caso.random() * (massimo - minimo + 1))


def _crea_parco_desktop():
    """Crea i 48 desktop finti. Il seme fisso (42) rende i dati identici su ogni PC."""
    caso = random.Random(42)
    adesso = datetime.now().replace(microsecond=0)
    costo_orario = {"piccola": 0.12, "media": 0.25, "grande": 0.48}
    taglie = ["piccola", "media", "media", "grande"]
    sedi = ["milano", "roma", "torino"]
    ambienti = ["produzione", "produzione", "collaudo"]
    parco = {}
    for numero in range(1, 49):
        # Stato e giorni trascorsi dall'ultimo accesso dipendono dal "tipo" di desktop.
        if numero in SPENTI:
            stato = "SPENTO"
            giorni = _intero(caso, 90, 300) if numero in SENZA_ASSEGNAZIONE else _intero(caso, 8, 25)
        elif numero in ACCESI_E_INUTILIZZATI:
            stato, giorni = "ACCESO", _intero(caso, 45, 200)
        else:
            stato, giorni = "ACCESO", _intero(caso, 0, 6)
        ultimo_accesso = adesso - timedelta(
            days=giorni, hours=_intero(caso, 1, 8), minutes=_intero(caso, 0, 59)
        )
        taglia = taglie[_intero(caso, 0, 3)]
        nome = f"vdi-{numero:04d}"
        parco[nome] = {
            "nome": nome,
            "stato": stato,
            "taglia": taglia,
            "costo_orario_eur": costo_orario[taglia],
            "ultimo_accesso": ultimo_accesso.isoformat(),
            "etichette": {
                "sede": sedi[_intero(caso, 0, 2)],
                "ambiente": ambienti[_intero(caso, 0, 2)],
            },
        }
    return parco


def _scrivi_file_assegnazioni(percorso):
    """Scrive il file CSV che dice quale matricola ha in carico quale desktop."""
    caso = random.Random(7)
    uffici = ["crediti", "tesoreria", "sportelli", "legale", "sistemi"]
    righe = []
    for numero in range(1, 49):
        if numero not in SENZA_ASSEGNAZIONE:
            riga = f"U{4100 + numero * 37};vdi-{numero:04d};{uffici[_intero(caso, 0, 4)]}"
            righe.append((caso.random(), riga))
    for posizione, nome in enumerate(ASSEGNAZIONI_SENZA_DESKTOP):
        riga = f"U{6200 + posizione * 51};{nome};{uffici[_intero(caso, 0, 4)]}"
        righe.append((caso.random(), riga))
    righe.sort()  # ordine "mescolato", ma sempre lo stesso
    testo = "matricola;nome_desktop;ufficio\n" + "\n".join(riga for _, riga in righe) + "\n"
    Path(percorso).write_text(testo, encoding="utf-8")


# ----------------------------------------------------------------------
# 2) La memoria del servizio
# ----------------------------------------------------------------------

class _StatoSimulatore:
    """Tutto ciò che il servizio finto ricorda mentre è acceso."""

    def __init__(self):
        self.blocco = threading.Lock()   # impedisce a due richieste di modificare i dati insieme
        self.parco = _crea_parco_desktop()
        self.token_validi = {}           # token -> nome utente
        self.operazioni = {}             # identificativo -> dati dell'operazione
        self.registro = []               # traccia di tutte le azioni richieste

    def annota(self, utente, azione, desktop, esito):
        """Aggiunge una riga al registro delle azioni."""
        self.registro.append({
            "quando": datetime.now().replace(microsecond=0).isoformat(),
            "utente": utente,
            "azione": azione,
            "desktop": desktop,
            "esito": esito,
        })

    def completa_operazioni_terminate(self):
        """Le operazioni il cui tempo è trascorso diventano FATTA e cambiano lo stato del desktop."""
        for operazione in self.operazioni.values():
            if operazione["stato"] == "IN_CORSO" and time.time() >= operazione["_termina_alle"]:
                operazione["stato"] = "FATTA"
                self.parco[operazione["desktop"]]["stato"] = operazione["_stato_finale"]


def _senza_campi_interni(dizionario):
    """Toglie i campi il cui nome inizia con '_': servono solo al simulatore."""
    return {chiave: valore for chiave, valore in dizionario.items() if not chiave.startswith("_")}


# ----------------------------------------------------------------------
# 3) Lo sportello: riceve le richieste HTTP e prepara le risposte
# ----------------------------------------------------------------------

class _GestoreRichieste(BaseHTTPRequestHandler):
    """Per ogni richiesta ricevuta decide che cosa rispondere, sempre in formato JSON."""

    def log_message(self, formato, *argomenti):
        pass  # non stampiamo i messaggi tecnici del server

    def _rispondi(self, codice, contenuto):
        corpo = json.dumps(contenuto, ensure_ascii=False).encode("utf-8")
        self.send_response(codice)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(corpo)))
        self.end_headers()
        self.wfile.write(corpo)

    def _utente_del_token(self):
        """Restituisce il nome utente legato al token ricevuto, oppure None."""
        intestazione = self.headers.get("Authorization", "")
        if not intestazione.startswith("Bearer "):
            return None
        return self.server.stato.token_validi.get(intestazione[len("Bearer "):].strip())

    def _non_autenticato(self):
        self._rispondi(401, {
            "errore": "Token mancante o non valido: occorre prima autenticarsi con POST /login"
        })

    # ---- richieste di lettura -------------------------------------------
    def do_GET(self):
        stato = self.server.stato
        indirizzo = urlparse(self.path)
        parti = [parte for parte in indirizzo.path.split("/") if parte]
        with stato.blocco:
            stato.completa_operazioni_terminate()
            if not parti:
                # Indirizzo "/": il biglietto da visita del servizio, senza autenticazione.
                return self._rispondi(200, {
                    "servizio": "Simulatore didattico di gestione dei desktop virtuali",
                    "avviso": "Tutti i dati sono inventati",
                    "endpoint": [
                        "POST /login",
                        "GET /desktop",
                        "GET /desktop/{nome}",
                        "POST /desktop/{nome}/spegni",
                        "POST /desktop/{nome}/accendi",
                        "GET /operazioni/{id}",
                        "GET /registro",
                    ],
                })
            if self._utente_del_token() is None:
                return self._non_autenticato()
            if parti == ["desktop"]:
                return self._elenco_desktop(parse_qs(indirizzo.query))
            if len(parti) == 2 and parti[0] == "desktop":
                desktop = stato.parco.get(parti[1])
                if desktop is None:
                    return self._rispondi(404, {"errore": f"Il desktop '{parti[1]}' non esiste"})
                return self._rispondi(200, desktop)
            if len(parti) == 2 and parti[0] == "operazioni":
                operazione = stato.operazioni.get(parti[1])
                if operazione is None:
                    return self._rispondi(404, {"errore": f"L'operazione '{parti[1]}' non esiste"})
                return self._rispondi(200, _senza_campi_interni(operazione))
            if parti == ["registro"]:
                return self._rispondi(200, {"elementi": stato.registro})
            self._rispondi(404, {"errore": f"L'indirizzo '{indirizzo.path}' non esiste"})

    def _elenco_desktop(self, parametri):
        """Restituisce una pagina dell'elenco dei desktop."""
        try:
            pagina = int(parametri.get("pagina", ["1"])[0])
            per_pagina = int(parametri.get("per_pagina", ["20"])[0])
        except ValueError:
            return self._rispondi(400, {
                "errore": "I parametri 'pagina' e 'per_pagina' devono essere numeri interi"
            })
        if pagina < 1 or not 1 <= per_pagina <= 25:
            return self._rispondi(400, {
                "errore": "'pagina' parte da 1; 'per_pagina' deve essere compreso tra 1 e 25"
            })
        tutti = list(self.server.stato.parco.values())
        pagine_totali = max(1, -(-len(tutti) // per_pagina))  # divisione arrotondata per eccesso
        inizio = (pagina - 1) * per_pagina
        self._rispondi(200, {
            "pagina": pagina,
            "per_pagina": per_pagina,
            "pagine_totali": pagine_totali,
            "totale": len(tutti),
            "elementi": tutti[inizio:inizio + per_pagina],
        })

    # ---- richieste che inviano dati o chiedono un'azione ------------------
    def do_POST(self):
        stato = self.server.stato
        parti = [parte for parte in urlparse(self.path).path.split("/") if parte]
        lunghezza = int(self.headers.get("Content-Length") or 0)
        testo = self.rfile.read(lunghezza).decode("utf-8", errors="replace") if lunghezza else ""
        try:
            dati = json.loads(testo) if testo else {}
        except ValueError:
            dati = None
        if not isinstance(dati, dict):
            return self._rispondi(400, {"errore": "Il corpo della richiesta deve essere un oggetto JSON"})
        with stato.blocco:
            stato.completa_operazioni_terminate()
            if parti == ["login"]:
                return self._login(dati)
            utente = self._utente_del_token()
            if utente is None:
                return self._non_autenticato()
            if len(parti) == 3 and parti[0] == "desktop" and parti[2] in ("spegni", "accendi"):
                return self._azione(utente, parti[1], parti[2])
            self._rispondi(404, {"errore": "Indirizzo o azione inesistente"})

    def _login(self, dati):
        """Controlla nome utente e password; se sono giusti consegna un token."""
        stato = self.server.stato
        nome = str(dati.get("utente", ""))
        password, ruolo = UTENZE_FINTE.get(nome, (None, None))
        if password is None or dati.get("password") != password:
            stato.annota(nome or "(sconosciuto)", "login", "-", "rifiutato: credenziali errate")
            return self._rispondi(401, {"errore": "Nome utente o password errati"})
        token = secrets.token_hex(16)
        stato.token_validi[token] = nome
        stato.annota(nome, "login", "-", "riuscito")
        self._rispondi(200, {"token": token, "utente": nome, "ruolo": ruolo})

    def _azione(self, utente, nome_desktop, azione):
        """Avvia lo spegnimento o l'accensione di un desktop, se l'utente ne ha il permesso."""
        stato = self.server.stato
        desktop = stato.parco.get(nome_desktop)
        if desktop is None:
            return self._rispondi(404, {"errore": f"Il desktop '{nome_desktop}' non esiste"})
        ruolo = UTENZE_FINTE[utente][1]
        if ruolo != "operativo":
            stato.annota(utente, azione, nome_desktop, "rifiutata: permessi insufficienti")
            return self._rispondi(403, {
                "errore": f"L'utente '{utente}' ha il ruolo '{ruolo}': può leggere, ma non eseguire azioni"
            })
        stato_finale = "SPENTO" if azione == "spegni" else "ACCESO"
        if desktop["stato"] == stato_finale:
            stato.annota(utente, azione, nome_desktop, "nessuna azione: stato già raggiunto")
            return self._rispondi(200, {
                "messaggio": f"Il desktop '{nome_desktop}' era già {stato_finale}: nessuna azione",
                "stato": stato_finale,
            })
        if desktop["stato"] not in ("ACCESO", "SPENTO"):
            return self._rispondi(409, {
                "errore": f"Sul desktop '{nome_desktop}' c'è già un'operazione in corso"
            })
        identificativo = f"op-{len(stato.operazioni) + 1:04d}"
        desktop["stato"] = "IN_SPEGNIMENTO" if azione == "spegni" else "IN_ACCENSIONE"
        stato.operazioni[identificativo] = {
            "id": identificativo,
            "azione": azione,
            "desktop": nome_desktop,
            "richiesta_da": utente,
            "stato": "IN_CORSO",
            "_termina_alle": time.time() + DURATA_OPERAZIONE,
            "_stato_finale": stato_finale,
        }
        stato.annota(utente, azione, nome_desktop, "accettata")
        self._rispondi(202, _senza_campi_interni(stato.operazioni[identificativo]))


class _ServizioLocale(ThreadingHTTPServer):
    """Il servizio vero e proprio: resta in ascolto e gestisce più richieste insieme."""

    daemon_threads = True

    def server_bind(self):
        # Versione ridotta dell'avvio: evita una ricerca del nome di rete
        # che su alcuni PC aziendali richiede parecchi secondi.
        socketserver.TCPServer.server_bind(self)
        self.server_name = "127.0.0.1"
        self.server_port = self.server_address[1]


# ----------------------------------------------------------------------
# 4) Accensione e spegnimento del simulatore
# ----------------------------------------------------------------------

if "_SIMULATORE_ATTIVO" not in globals():
    _SIMULATORE_ATTIVO = None


def ferma_simulatore():
    """Spegne il simulatore, se è acceso."""
    global _SIMULATORE_ATTIVO
    if _SIMULATORE_ATTIVO is not None:
        _SIMULATORE_ATTIVO.shutdown()
        _SIMULATORE_ATTIVO.server_close()
        _SIMULATORE_ATTIVO = None


def avvia_simulatore(cartella="."):
    """Avvia il servizio finto con i dati iniziali.

    Restituisce due valori: l'indirizzo a cui il servizio risponde
    e il percorso del file CSV delle assegnazioni.
    Se il simulatore era già acceso, lo riavvia: i dati tornano quelli iniziali.
    """
    global _SIMULATORE_ATTIVO
    ferma_simulatore()
    # "127.0.0.1" = solo questo PC; porta 0 = "scegli tu una porta libera".
    servizio = _ServizioLocale(("127.0.0.1", 0), _GestoreRichieste)
    servizio.stato = _StatoSimulatore()
    threading.Thread(target=servizio.serve_forever, daemon=True).start()
    _SIMULATORE_ATTIVO = servizio
    file_assegnazioni = Path(cartella) / "assegnazioni.csv"
    _scrivi_file_assegnazioni(file_assegnazioni)
    return f"http://127.0.0.1:{servizio.server_address[1]}", file_assegnazioni


print("Simulatore pronto: ora va avviato con la cella successiva.")

### 3.3 Avviamo il servizio

La funzione `avvia_simulatore()` accende il servizio e ci restituisce due valori: l'indirizzo a cui il servizio risponde e il percorso del file delle assegnazioni. Li conserviamo in due variabili, che useremo in tutto il laboratorio.

In [ ]:
# La funzione restituisce due valori: li riceviamo in due variabili, separate da una virgola.
URL_BASE, FILE_ASSEGNAZIONI = avvia_simulatore()

print("Il servizio risponde all'indirizzo:", URL_BASE)
print("File delle assegnazioni           :", FILE_ASSEGNAZIONI.resolve())

**Come leggere il risultato.** L'indirizzo ha la forma `http://127.0.0.1:` seguita da un numero. Lo leggiamo in tre parti.

| Parte | Significato |
|---|---|
| `http://` | Il protocollo con cui parliamo al servizio |
| `127.0.0.1` | Un indirizzo speciale che significa «questo stesso computer». È detto anche *localhost* |
| il numero dopo i due punti | La **porta**: un numero che distingue i diversi servizi in ascolto sullo stesso computer, un po' come l'interno di un numero di telefono |

La porta viene scelta al momento dell'avvio tra quelle libere, quindi è diversa da un PC all'altro. Per questo non la scriviamo mai a mano: usiamo sempre la variabile `URL_BASE`.

**Da ricordare.** Se rieseguiamo questa cella, il simulatore riparte da capo con i dati iniziali. È il modo per rifare il laboratorio dall'inizio. Dopo un riavvio dobbiamo ripetere anche l'accesso della sezione 5.

---
## 4. La prima richiesta

### 4.1 Prepariamo la sessione

Con `requests` potremmo inviare ogni richiesta in modo indipendente dalle altre. È più comodo creare una **sessione**: un oggetto che ricorda le impostazioni comuni a tutte le richieste, così non dobbiamo ripeterle ogni volta.

Nella cella impostiamo anche due cose.

- **Niente proxy.** Nelle reti aziendali le richieste dirette a Internet passano di solito da un computer intermedio, chiamato *proxy*, che le controlla. Il proxy non sa nulla del nostro simulatore, che vive dentro il nostro PC. Con `trust_env = False` diciamo a `requests` di collegarsi direttamente. Con un servizio reale esterno questa riga di norma va tolta, perché in quel caso il proxy serve.
- **Un tempo massimo di attesa.** Se il servizio non risponde, non vogliamo che il programma resti bloccato per sempre. Stabiliamo che ogni richiesta attenda al massimo 10 secondi.

In [ ]:
# Creiamo la sessione: da qui in poi tutte le richieste passeranno da questo oggetto.
sessione = requests.Session()

# Il simulatore è sul nostro PC: ci colleghiamo direttamente, senza passare dal proxy aziendale.
sessione.trust_env = False

# Tempo massimo di attesa per ogni risposta, in secondi.
ATTESA_MASSIMA = 10

print("Sessione pronta.")

### 4.2 Bussiamo alla porta

Inviamo la richiesta più semplice possibile: una richiesta di tipo **GET** all'indirizzo principale del servizio.

Ogni richiesta HTTP ha un *tipo*. A noi ne servono due:

- **GET** serve a **leggere**. Non modifica nulla;
- **POST** serve a **inviare dati** o a **chiedere un'azione**.

Nella documentazione delle API il tipo di richiesta è chiamato *metodo HTTP*. Non va confuso con i metodi di Python, che incontriamo tra poco.

In [ ]:
# Richiesta GET all'indirizzo principale del servizio: URL_BASE seguito da "/".
risposta = sessione.get(URL_BASE + "/", timeout=ATTESA_MASSIMA)

# Ogni risposta HTTP porta con sé un "codice di stato": un numero che dice com'è andata.
print("Codice di stato:", risposta.status_code)

# Il contenuto della risposta è un testo.
print("Testo ricevuto :")
print(risposta.text)

**Come leggere il risultato.**

- Il **codice di stato** `200` significa «tutto bene». Tra poco vedremo gli altri codici.
- Il testo ricevuto è scritto in formato **JSON** (*JavaScript Object Notation*). È il formato di testo più usato per scambiare dati tra programmi. Somiglia molto ai dizionari e alle liste di Python: le parentesi graffe `{ }` racchiudono coppie «nome: valore», le parentesi quadre `[ ]` racchiudono elenchi.

Il testo, così com'è, è scomodo da usare. Il metodo `.json()` della risposta lo trasforma in oggetti Python: le graffe diventano un dizionario, le quadre una lista. In Python un **metodo** è una funzione che appartiene a un oggetto: si richiama scrivendo un punto dopo il nome dell'oggetto, come in `risposta.json()`.

In [ ]:
# Trasformiamo il testo JSON in oggetti Python.
dati = risposta.json()

print("Tipo dell'oggetto ottenuto:", type(dati).__name__)
print("Nome del servizio         :", dati["servizio"])
print()
print("Operazioni che il servizio mette a disposizione:")
for voce in dati["endpoint"]:
    print("  -", voce)

**Come leggere il risultato.** Il servizio ci ha consegnato l'elenco delle proprie operazioni. Ogni riga è un **endpoint**: un indirizzo a cui il servizio risponde per una certa operazione. Nei sistemi reali questo elenco si trova nella documentazione della API.

| Endpoint | Che cosa fa |
|---|---|
| `POST /login` | Riceve nome utente e password, restituisce il lasciapassare |
| `GET /desktop` | Restituisce l'elenco dei desktop |
| `GET /desktop/{nome}` | Restituisce i dati di un solo desktop |
| `POST /desktop/{nome}/spegni` | Avvia lo spegnimento di un desktop |
| `POST /desktop/{nome}/accendi` | Avvia l'accensione di un desktop |
| `GET /operazioni/{id}` | Dice a che punto è un'operazione avviata |
| `GET /registro` | Restituisce la traccia di tutte le azioni richieste |

Le parti tra parentesi graffe, come `{nome}`, sono segnaposto: al loro posto scriviamo un valore vero, per esempio `/desktop/vdi-0007`.

### 4.3 Chiediamo l'elenco dei desktop, senza presentarci

Proviamo subito a leggere l'elenco dei desktop. Non abbiamo ancora detto al servizio chi siamo: vediamo come reagisce.

In [ ]:
risposta = sessione.get(URL_BASE + "/desktop", timeout=ATTESA_MASSIMA)

print("Codice di stato:", risposta.status_code)
print("Risposta       :", risposta.json())

**Come leggere il risultato.** Il servizio ha risposto con il codice `401` e con un messaggio di errore: non sa chi siamo, quindi non ci consegna nulla. Non è un guasto. È lo sportello che fa il suo mestiere.

Il codice di stato è il primo elemento da guardare in ogni risposta. Ecco i codici che incontreremo.

| Codice | Significato | Dove lo incontriamo |
|---|---|---|
| `200` | Tutto bene | Lettura riuscita |
| `202` | Richiesta accettata, lavoro in corso | Spegnimento avviato |
| `400` | Richiesta scritta male | Parametri sbagliati |
| `401` | Non autenticati: il servizio non sa chi siamo | Richiesta senza lasciapassare |
| `403` | Autenticati, ma non autorizzati a quella operazione | Il lettore prova a spegnere |
| `404` | La cosa richiesta non esiste | Desktop inesistente |
| `409` | Richiesta in conflitto con lo stato attuale | Azione su un desktop che sta già cambiando stato |

Una regola pratica: i codici che iniziano con **2** dicono che è andata bene. Quelli che iniziano con **4** segnalano un errore nella nostra richiesta. Quelli che iniziano con **5** segnalano un problema dalla parte del servizio.

---
## 5. Passo 1 – Autenticarsi

### 5.1 Autenticazione e autorizzazione

Sono due parole simili, che indicano due controlli diversi.

- L'**autenticazione** risponde alla domanda «chi siamo?». È il controllo dell'identità.
- L'**autorizzazione** risponde alla domanda «che cosa ci è permesso fare?». Viene dopo: prima il servizio ci riconosce, poi verifica i nostri permessi.

I due codici di errore visti poco fa corrispondono ai due controlli: `401` significa che manca l'autenticazione, `403` che manca l'autorizzazione.

### 5.2 Il lasciapassare: il token

Molti servizi non vogliono ricevere la password a ogni richiesta. Funzionano così:

1. presentiamo nome utente e password **una volta sola**, a un endpoint apposito, che nel nostro caso è `/login`;
2. il servizio controlla e ci restituisce un **token**: una lunga sequenza casuale di caratteri, che vale come lasciapassare temporaneo;
3. da quel momento alleghiamo il token a ogni richiesta, al posto della password.

I vantaggi sono due. La password viaggia una volta sola. Il token, a differenza della password, dura poco: nei servizi reali scade dopo qualche minuto o qualche ora, e può essere revocato in qualunque momento. Nel nostro simulatore, per semplicità, il token resta valido finché il simulatore è acceso.

### 5.3 Dove teniamo la password

Prima di autenticarci dobbiamo decidere dove scrivere nome utente e password. Vediamo prima il modo più diffuso, poi quello consigliato.

**Il modo più diffuso (sconsigliato).** Scriviamo la password direttamente nel codice.

In [ ]:
# SCONSIGLIATO: la password è scritta nel codice.
# Chiunque apra il notebook, o lo riceva per posta, la legge.
utente = "lettore"
password = "aula-lettura"

**Il modo consigliato.** Leggiamo la password da una **variabile d'ambiente**: un valore conservato dal sistema operativo, fuori dal codice. Chi prepara il PC o il server imposta la variabile. Il programma si limita a leggerla.

In Python le variabili d'ambiente si leggono con `os.environ.get(nome, valore_di_riserva)`. Se la variabile non esiste, la funzione restituisce il valore di riserva.

In [ ]:
# CONSIGLIATO: nome utente e password si leggono da variabili d'ambiente.
# Il secondo argomento è il valore di riserva, usato se la variabile non è impostata.
utente = os.environ.get("VDI_UTENTE", "lettore")
password = os.environ.get("VDI_PASSWORD", "aula-lettura")

print("Utente con cui ci presenteremo:", utente)

**Una precisazione importante.** In aula le due variabili d'ambiente non sono impostate, quindi Python usa i valori di riserva. Possiamo permettercelo solo perché la password è finta e pubblica. In un caso reale il valore di riserva non ci sarebbe: senza la variabile il programma si fermerebbe, ed è giusto così.

Nelle organizzazioni più strutturate le password dei programmi non stanno nemmeno nelle variabili d'ambiente, ma in un archivio protetto apposito, chiamato *gestore dei segreti*. Il principio resta lo stesso: **le credenziali vivono fuori dal codice**.

### 5.4 Chiediamo il token

Inviamo nome utente e password all'endpoint `/login`. Questa volta la richiesta è di tipo **POST**, perché inviamo dati. L'argomento `json=` trasforma il nostro dizionario Python in testo JSON e lo allega alla richiesta.

In [ ]:
risposta = sessione.post(
    URL_BASE + "/login",
    json={"utente": utente, "password": password},
    timeout=ATTESA_MASSIMA,
)
print("Codice di stato:", risposta.status_code)

# La risposta contiene il token e alcune informazioni su di noi.
dati_login = risposta.json()
token = dati_login["token"]

print("Utente riconosciuto:", dati_login["utente"])
print("Ruolo assegnato    :", dati_login["ruolo"])
print("Token (inizio)     :", token[:8] + "...")

**Come leggere il risultato.** Il codice `200` conferma che il servizio ci ha riconosciuti. Ci ha assegnato il ruolo `lettura` e ci ha consegnato il token.

Del token stampiamo solo i primi caratteri. Chi possiede il token può agire a nome nostro finché il token è valido: va trattato con la stessa riservatezza di una password.

### 5.5 Usiamo il token

Ogni richiesta HTTP porta con sé, oltre all'indirizzo, alcune righe di informazioni di servizio chiamate **intestazioni** (in inglese *headers*). Il token viaggia nell'intestazione `Authorization`, preceduto dalla parola `Bearer`, che significa «portatore»: chi porta questo token ha diritto di passare.

Ripetiamo la richiesta che prima era stata respinta, allegando l'intestazione.

In [ ]:
# Costruiamo l'intestazione con il token.
intestazioni = {"Authorization": "Bearer " + token}

# La stessa richiesta di prima, questa volta con l'intestazione.
risposta = sessione.get(URL_BASE + "/desktop", headers=intestazioni, timeout=ATTESA_MASSIMA)
print("Codice di stato:", risposta.status_code)

Il codice è passato da `401` a `200`: il servizio ci ha riconosciuti e ha risposto.

Scrivere `headers=intestazioni` in ogni richiesta sarebbe noioso, e prima o poi lo dimenticheremmo. Memorizziamo allora l'intestazione nella sessione: da qui in poi verrà allegata automaticamente.

In [ ]:
# Da questo momento ogni richiesta della sessione porterà con sé il token.
sessione.headers["Authorization"] = "Bearer " + token

# Verifica: la stessa richiesta, senza indicare le intestazioni.
risposta = sessione.get(URL_BASE + "/desktop", timeout=ATTESA_MASSIMA)
print("Codice di stato:", risposta.status_code)

### 5.6 Racchiudiamo l'accesso in una funzione

Dovremo autenticarci più volte, anche con un'utenza diversa. Raccogliamo i passaggi in una funzione.

Il testo tra triple virgolette, subito sotto la prima riga della funzione, è la **docstring**: la descrizione di ciò che la funzione fa, scritta per chi la userà.

Nella funzione compare una novità: `raise_for_status()`. Questo metodo controlla il codice di stato della risposta. Se il codice segnala un errore, cioè se inizia con 4 o con 5, interrompe il programma con un messaggio chiaro. È un'abitudine preziosa: **meglio fermarsi subito che proseguire con dati sbagliati**.

In [ ]:
def accedi(utente, password):
    """Si autentica presso il servizio e memorizza il token nella sessione.

    Restituisce il ruolo che il servizio ha assegnato all'utente.
    """
    risposta = sessione.post(
        URL_BASE + "/login",
        json={"utente": utente, "password": password},
        timeout=ATTESA_MASSIMA,
    )
    # Se l'accesso è stato rifiutato, ci fermiamo qui con un errore chiaro.
    risposta.raise_for_status()

    dati = risposta.json()
    sessione.headers["Authorization"] = "Bearer " + dati["token"]
    return dati["ruolo"]


ruolo = accedi(utente, password)
print("Accesso eseguito con il ruolo:", ruolo)

> **Proviamo noi – Esercizio 1.** Nella cella vuota qui sotto proviamo ad accedere con una password sbagliata, scrivendo `accedi("lettore", "sbagliata")`. Che cosa succede? Quale codice di stato compare nel messaggio di errore?

<details>
<summary>Soluzione</summary>

```python
accedi("lettore", "sbagliata")
```

Il programma si ferma con un errore di tipo `HTTPError` e con il messaggio `401 Client Error`. È l'effetto di `raise_for_status()`: l'accesso è stato rifiutato, e la funzione non prosegue.

La sessione conserva il token ottenuto in precedenza, quindi le richieste successive continuano a funzionare.

</details>

In [ ]:
# Spazio per l'esercizio 1.

---
## 6. Passo 2 – Leggere l'inventario

### 6.1 La prima risposta

Ora che siamo autenticati, guardiamo con attenzione che cosa ci consegna l'endpoint `/desktop`.

In [ ]:
risposta = sessione.get(URL_BASE + "/desktop", timeout=ATTESA_MASSIMA)
risposta.raise_for_status()
pagina = risposta.json()

# La risposta è un dizionario: vediamo quali chiavi contiene.
print("Chiavi della risposta   :", list(pagina.keys()))
print("Desktop in questa pagina:", len(pagina["elementi"]))
print("Desktop totali          :", pagina["totale"])
print("Pagine totali           :", pagina["pagine_totali"])

**Come leggere il risultato.** Abbiamo ricevuto 20 desktop, ma il servizio dichiara che in totale sono 48, distribuiti su 3 pagine.

Questo meccanismo si chiama **paginazione**. Quando un elenco è lungo, il servizio non lo consegna tutto insieme: lo divide in blocchi, chiamati *pagine*, e ne restituisce uno per richiesta. Lo fanno quasi tutte le API, per non sovraccaricare né il servizio né chi riceve.

È anche **l'errore più frequente** di chi comincia: fermarsi alla prima pagina, credendo di avere tutto. Un controllo fatto su 20 desktop anziché su 48 darebbe risultati sbagliati, senza alcun messaggio di errore.

Prima di leggere le altre pagine, guardiamo com'è fatto un singolo desktop.

In [ ]:
# Prendiamo il primo desktop della pagina: è un dizionario.
primo = pagina["elementi"][0]

# json.dumps trasforma il dizionario in testo; indent=2 lo dispone su più righe, ben allineato.
# ensure_ascii=False lascia leggibili le eventuali lettere accentate.
print(json.dumps(primo, indent=2, ensure_ascii=False))

**Come leggere il risultato.** Ogni desktop è descritto da sei campi.

| Campo | Significato |
|---|---|
| `nome` | Il nome del desktop, che lo identifica |
| `stato` | `ACCESO` oppure `SPENTO` |
| `taglia` | La dimensione della macchina: `piccola`, `media` o `grande` |
| `costo_orario_eur` | Il costo in euro per ogni ora in cui il desktop resta acceso (valore inventato) |
| `ultimo_accesso` | Data e ora dell'ultima volta in cui un utente ha usato il desktop |
| `etichette` | Informazioni aggiuntive assegnate da chi gestisce i desktop: la sede e l'ambiente. L'ambiente è `produzione` per i desktop usati nel lavoro di ogni giorno, `collaudo` per quelli usati nelle prove |

Due osservazioni.

- La data è scritta nella forma `anno-mese-giorno`, poi la lettera `T`, poi `ore:minuti:secondi`. È il formato internazionale **ISO 8601**, usato quasi sempre nelle API perché non lascia dubbi tra giorno e mese.
- Il campo `etichette` contiene a sua volta un dizionario. Quando un dizionario ne contiene un altro parliamo di dati **annidati**. Tra poco vedremo come trattarli.

### 6.2 Leggiamo tutte le pagine

Per chiedere una pagina precisa aggiungiamo alla richiesta due **parametri**: `pagina`, che indica quale pagina vogliamo, e `per_pagina`, che indica quanti desktop vogliamo in ogni pagina. Con `requests` i parametri si passano come dizionario, con l'argomento `params=`.

Scriviamo una funzione che chiede le pagine una dopo l'altra e si ferma quando ha letto l'ultima.

In [ ]:
def leggi_tutti_i_desktop():
    """Legge l'inventario completo, una pagina dopo l'altra.

    Restituisce una lista di dizionari, uno per ogni desktop.
    """
    tutti = []           # qui accumuliamo i desktop di tutte le pagine
    numero_pagina = 1    # partiamo dalla prima pagina

    while True:
        risposta = sessione.get(
            URL_BASE + "/desktop",
            params={"pagina": numero_pagina, "per_pagina": 20},
            timeout=ATTESA_MASSIMA,
        )
        risposta.raise_for_status()
        pagina = risposta.json()

        # Aggiungiamo alla lista i desktop di questa pagina.
        tutti.extend(pagina["elementi"])

        # Se questa era l'ultima pagina, usciamo dal ciclo.
        if numero_pagina >= pagina["pagine_totali"]:
            break

        # Altrimenti passiamo alla pagina successiva.
        numero_pagina = numero_pagina + 1

    return tutti


elenco_desktop = leggi_tutti_i_desktop()
print("Desktop letti:", len(elenco_desktop))

**Come leggere il risultato.** Ora i desktop sono 48, come dichiarato dal servizio. Confrontare il numero di elementi letti con il totale dichiarato è un controllo semplice, che conviene fare sempre.

> **Proviamo noi – Esercizio 2.** Nella funzione cambiamo `"per_pagina": 20` in `"per_pagina": 10` e rieseguiamo la cella. I desktop letti cambiano? Quante richieste servono ora? Poi proviamo con `100`: che cosa succede? Alla fine rimettiamo `20` e rieseguiamo la cella.

<details>
<summary>Soluzione</summary>

Con `10` i desktop letti restano 48, ma servono 5 richieste invece di 3.

Con `100` il programma si ferma con l'errore `400 Client Error`: il servizio accetta al massimo 25 desktop per pagina. Molte API impongono un limite di questo tipo. Lo si trova nella documentazione oppure, come qui, nel messaggio di errore del servizio.

</details>

### 6.3 Dalla lista di dizionari alla tabella

Una lista di 48 dizionari è scomoda da interrogare. Per contare, filtrare e incrociare ci serve una tabella, cioè un **DataFrame** di pandas.

**Primo tentativo.** Usiamo `pd.DataFrame`, che costruisce una tabella da una lista di dizionari: ogni dizionario diventa una riga, ogni chiave una colonna.

In [ ]:
tabella_grezza = pd.DataFrame(elenco_desktop)

# .head() mostra le prime cinque righe.
tabella_grezza.head()

**Come leggere il risultato.** La tabella c'è, ma la colonna `etichette` contiene ancora un dizionario in ogni riga. In questa forma non possiamo, per esempio, contare i desktop di una sede.

**Secondo tentativo.** Usiamo `pd.json_normalize`, che è fatto apposta per i dati annidati: li «appiattisce». Ogni chiave del dizionario interno diventa una colonna a sé, con un nome composto, per esempio `etichette.sede`.

In [ ]:
# json_normalize appiattisce i dati annidati: una colonna per ogni chiave interna.
inventario = pd.json_normalize(elenco_desktop)

# Rinominiamo le due colonne con nomi più comodi.
inventario = inventario.rename(columns={
    "etichette.sede": "sede",
    "etichette.ambiente": "ambiente",
})

inventario.head()

### 6.4 Sistemiamo le date

La colonna `ultimo_accesso` è arrivata come testo. Per fare calcoli con le date dobbiamo convertirla in un vero tipo «data e ora». Poi calcoliamo da quanti giorni ciascun desktop non viene usato: è il dato che ci servirà per la terza domanda del responsabile.

In [ ]:
# Convertiamo il testo in data e ora.
inventario["ultimo_accesso"] = pd.to_datetime(inventario["ultimo_accesso"])

# Data e ora di questo momento.
adesso = pd.Timestamp.now()

# La differenza tra due date è un intervallo di tempo; con .dt.days ne prendiamo i giorni interi.
inventario["giorni_inattivo"] = (adesso - inventario["ultimo_accesso"]).dt.days

inventario[["nome", "stato", "ultimo_accesso", "giorni_inattivo"]].head()

### 6.5 Un primo sguardo all'inventario

Con la tabella pronta bastano poche parole per ottenere un conteggio. Il metodo `value_counts()` conta quante volte compare ciascun valore di una colonna.

In [ ]:
# Quanti desktop sono accesi e quanti spenti?
inventario["stato"].value_counts()

**Come leggere il risultato.** Su 48 desktop, 43 sono accesi e 5 sono spenti.

> **Proviamo noi – Esercizio 3.** Quanti desktop ci sono in ciascuna sede? E quanti in ciascun ambiente?

<details>
<summary>Soluzione</summary>

```python
inventario["sede"].value_counts()
```

```python
inventario["ambiente"].value_counts()
```

</details>

In [ ]:
# Spazio per l'esercizio 3.

---
## 7. La seconda fonte: il file delle assegnazioni

Il servizio sa quali desktop esistono, ma non sa a chi sono assegnati. Questa informazione sta nel file `assegnazioni.csv`, che nel nostro scenario è tenuto da un altro ufficio.

Un file **CSV** (*Comma-Separated Values*, «valori separati da virgola») è un file di testo in cui ogni riga è una riga di tabella, e le colonne sono divise da un carattere separatore. In Italia il separatore è spesso il **punto e virgola**, perché la virgola serve già per i numeri decimali. Lo indichiamo a pandas con `sep=";"`.

In [ ]:
# Leggiamo il file CSV, indicando il punto e virgola come separatore di colonna.
assegnazioni = pd.read_csv(FILE_ASSEGNAZIONI, sep=";")

print("Righe lette:", len(assegnazioni))
assegnazioni.head()

**Come leggere il risultato.** Il file ha 44 righe e tre colonne: la **matricola** della persona (inventata), il **nome del desktop** che ha in carico e il suo **ufficio**.

Notiamo già un indizio: i desktop sono 48, le assegnazioni 44. I due elenchi non coincidono.

Per incrociare due tabelle serve una colonna che le colleghi, chiamata **chiave**. Qui la chiave è il nome del desktop, che però nelle due tabelle si chiama in modo diverso: `nome` nell'inventario, `nome_desktop` nelle assegnazioni. Uniformiamo il nome.

In [ ]:
# Diamo alla colonna chiave lo stesso nome che ha nell'inventario.
assegnazioni = assegnazioni.rename(columns={"nome_desktop": "nome"})

assegnazioni.head(3)

**Un'avvertenza per i dati reali.** Il nostro file è pulito. I file veri spesso non lo sono: lo stesso desktop può comparire come `VDI-0007`, come `vdi-0007` o con uno spazio in fondo. Per pandas sono tre valori diversi, e l'incrocio fallirebbe in silenzio.

Prima di incrociare conviene sempre ripulire la chiave, per esempio con `assegnazioni["nome"].str.strip().str.lower()`, che toglie gli spazi ai lati e porta tutto in minuscolo.

---
## 8. Passo 3 – Decidere

### 8.1 Incrociamo le due tabelle

Il metodo `merge` unisce due tabelle, affiancando le righe che hanno lo stesso valore nella colonna chiave. Usiamo due opzioni.

- `how="outer"` chiede di tenere **tutte** le righe di entrambe le tabelle, anche quelle che non trovano corrispondenza nell'altra. È proprio ciò che ci serve: le righe senza corrispondenza sono le anomalie che cerchiamo.
- `indicator=True` aggiunge una colonna, chiamata `_merge`, che per ogni riga dice da dove proviene.

| Valore di `_merge` | Significato | Nel nostro caso |
|---|---|---|
| `both` | La riga è presente in entrambe le tabelle | Desktop esistente e assegnato: tutto regolare |
| `left_only` | La riga è presente solo nella prima tabella | Desktop esistente, ma non assegnato a nessuno |
| `right_only` | La riga è presente solo nella seconda tabella | Assegnazione che riguarda un desktop inesistente |

In [ ]:
confronto = inventario.merge(
    assegnazioni,
    on="nome",         # la colonna chiave, presente in entrambe le tabelle
    how="outer",       # teniamo tutte le righe, anche quelle senza corrispondenza
    indicator=True,    # aggiunge la colonna "_merge", che dice da dove viene ogni riga
)

# Traduciamo i valori della colonna "_merge" in parole nostre.
# .astype(str) trasforma i valori in testo; .map(...) sostituisce ogni valore con la sua traduzione.
traduzione = {
    "both": "in entrambe",
    "left_only": "solo nell'inventario",
    "right_only": "solo nelle assegnazioni",
}
confronto["situazione"] = confronto["_merge"].astype(str).map(traduzione)

# La colonna "_merge" non ci serve più: la togliamo.
confronto = confronto.drop(columns="_merge")

confronto["situazione"].value_counts()

**Come leggere il risultato.** In tre righe abbiamo il quadro completo.

- **42** desktop sono presenti in entrambe le fonti: esistono e sono assegnati.
- **6** desktop compaiono solo nell'inventario: esistono, ma nessuno li ha in carico.
- **2** righe compaiono solo nelle assegnazioni: qualcuno risulta assegnatario di un desktop che non esiste.

Abbiamo già la risposta alle prime due domande del responsabile. Guardiamole nel dettaglio.

### 8.2 Prima domanda: desktop senza assegnazione

Per selezionare alcune righe di una tabella scriviamo una condizione tra parentesi quadre: restano solo le righe in cui la condizione è vera.

In [ ]:
# Teniamo solo le righe in cui la situazione è "solo nell'inventario".
senza_assegnazione = confronto[confronto["situazione"] == "solo nell'inventario"]

senza_assegnazione[["nome", "stato", "sede", "giorni_inattivo", "costo_orario_eur"]]

**Come leggere il risultato.** I desktop senza assegnazione sono sei, e non sono tutti uguali.

- Due sono **spenti**: non costano, ma occupano un posto nell'inventario. Vanno segnalati, per capire se dismetterli.
- Tre sono **accesi e fermi da mesi**: nessuno li usa, eppure costano.
- Uno, `vdi-0027`, è **acceso ed è stato usato pochi giorni fa**. Qualcuno lo sta usando, anche se non risulta assegnato a nessuno.

L'ultimo caso è il più istruttivo. Uno script può trovare l'anomalia, ma non può sapere se dietro c'è un errore nel file oppure un uso non previsto. **Non tutto ciò che è anomalo va corretto in automatico**: alcune anomalie vanno soltanto segnalate a una persona.

**Un dettaglio sui numeri.** Nella colonna `giorni_inattivo` i valori ora compaiono con un decimale, per esempio `194.0`, mentre nell'inventario erano numeri interi. Dopo l'incrocio la colonna contiene anche due valori mancanti: quelli delle due righe che non hanno un desktop. Per poter segnare un valore mancante, pandas trasforma l'intera colonna da numeri interi a numeri decimali. I valori non cambiano: cambia solo il modo in cui sono scritti. Li riporteremo a numeri interi più avanti, nel report.

### 8.3 Seconda domanda: assegnazioni senza desktop

Selezioniamo allo stesso modo le righe che compaiono solo nel file delle assegnazioni.

In [ ]:
# Teniamo solo le righe in cui la situazione è "solo nelle assegnazioni".
senza_desktop = confronto[confronto["situazione"] == "solo nelle assegnazioni"]

senza_desktop[["nome", "matricola", "ufficio", "stato"]]

**Come leggere il risultato.** Due persone risultano assegnatarie di desktop che il servizio non conosce. Forse i desktop sono stati eliminati e il file non è stato aggiornato.

Nella colonna `stato` compare `NaN`. È la sigla di *Not a Number* ed è il modo in cui pandas indica un **valore mancante**: per queste due righe l'inventario non ha nulla da dire, perché il desktop non c'è.

### 8.4 Terza domanda: desktop accesi ma inutilizzati

Cerchiamo i desktop che soddisfano **due condizioni insieme**: sono accesi, e l'ultimo accesso risale a più di 30 giorni fa.

Ogni condizione, applicata a una colonna, produce una **maschera**: una colonna di valori «vero» o «falso», uno per riga. Le due maschere si combinano con il simbolo `&`, che significa «e»: il risultato è vero solo dove sono vere entrambe.

La soglia dei 30 giorni è una nostra scelta. La teniamo in una variabile con un nome chiaro, così è facile trovarla e cambiarla.

In [ ]:
# La soglia, in giorni.
SOGLIA_GIORNI = 30

# Prima maschera: vero dove il desktop è acceso.
e_acceso = confronto["stato"] == "ACCESO"

# Seconda maschera: vero dove il desktop è fermo da più giorni della soglia.
e_inutilizzato = confronto["giorni_inattivo"] > SOGLIA_GIORNI

# Teniamo le righe in cui sono vere entrambe le condizioni.
# .copy() crea una tabella indipendente, che potremo modificare liberamente.
da_spegnere = confronto[e_acceso & e_inutilizzato].copy()

# In questa tabella non ci sono valori mancanti: riportiamo i giorni a numeri interi.
da_spegnere["giorni_inattivo"] = da_spegnere["giorni_inattivo"].astype(int)

print("Desktop accesi e inutilizzati da più di", SOGLIA_GIORNI, "giorni:", len(da_spegnere))

Ne abbiamo trovati 8. Calcoliamo quanto costano. Un desktop sempre acceso costa, in un mese, il suo costo orario moltiplicato per 24 ore e per 30 giorni.

In [ ]:
# Ore in un mese di 30 giorni.
ORE_IN_UN_MESE = 24 * 30

# Nuova colonna con il costo mensile di ciascun desktop.
# .round(2) arrotonda ai centesimi: nei calcoli con i decimali il computer può lasciare
# scarti minuscoli, per esempio 86.39999999999999 al posto di 86.4.
da_spegnere["costo_mensile_eur"] = (da_spegnere["costo_orario_eur"] * ORE_IN_UN_MESE).round(2)

# Ordiniamo dal desktop fermo da più tempo a quello fermo da meno tempo.
da_spegnere = da_spegnere.sort_values("giorni_inattivo", ascending=False)

# Le colonne che vogliamo mostrare e salvare.
colonne_report = ["nome", "situazione", "matricola", "ufficio", "sede", "giorni_inattivo", "costo_mensile_eur"]

da_spegnere[colonne_report]

In [ ]:
costo_totale = da_spegnere["costo_mensile_eur"].sum()

# La lettera f davanti alle virgolette permette di inserire nel testo il valore di una variabile,
# scritta tra parentesi graffe. La sigla :.2f significa "numero con due cifre decimali".
print(f"Costo mensile evitabile: {costo_totale:.2f} euro")

**Come leggere il risultato.** Gli 8 desktop comprendono i tre senza assegnazione visti prima e cinque desktop regolarmente assegnati, ma fermi da tempo. Il desktop `vdi-0027`, usato di recente, giustamente non compare.

Il costo è calcolato su prezzi inventati, ma il ragionamento è reale: **un numero in euro** è spesso l'argomento più convincente per ottenere l'autorizzazione ad agire.

> **Proviamo noi – Esercizio 4.** Cambiamo `SOGLIA_GIORNI` in `60`, rieseguiamo le tre celle di codice del paragrafo 8.4 e osserviamo come cambiano il numero di desktop e il costo. Poi proviamo con `90`. Alla fine rimettiamo `30` e rieseguiamo le tre celle, perché le sezioni successive usano quel valore.

<details>
<summary>Soluzione</summary>

Con la soglia a 60 giorni i desktop sono 6. Con la soglia a 90 giorni sono 3.

La soglia non è un dettaglio tecnico: è una **decisione di gestione**, che spetta al responsabile. Il programma la rende esplicita e facile da cambiare.

</details>

### 8.5 Il report

Riassumiamo le tre risposte in una piccola tabella. Poi salviamo l'elenco dei desktop da spegnere in un file che si apre con Excel.

In [ ]:
# Una riga per ciascuna delle tre domande del responsabile.
riepilogo = pd.DataFrame([
    {"controllo": "Desktop senza assegnazione", "quanti": len(senza_assegnazione)},
    {"controllo": "Assegnazioni senza desktop", "quanti": len(senza_desktop)},
    {"controllo": "Desktop accesi e inutilizzati", "quanti": len(da_spegnere)},
])

riepilogo

In [ ]:
# Salviamo il report in un file CSV che Excel in italiano apre senza problemi.
da_spegnere[colonne_report].to_csv(
    "report_desktop_da_spegnere.csv",
    sep=";",                 # punto e virgola come separatore di colonna
    decimal=",",             # virgola come separatore dei decimali
    index=False,             # non scriviamo la colonna con i numeri di riga
    encoding="utf-8-sig",    # fa riconoscere a Excel le lettere accentate
)

print("Report salvato nel file report_desktop_da_spegnere.csv")

A questo punto abbiamo **letto** e **deciso**, senza modificare nulla. Potremmo fermarci qui: un report affidabile, prodotto in pochi secondi e ripetibile, è già un risultato utile. Molte automazioni, nella pratica, si fermano proprio a questo punto.

---
## 9. Passo 4 – Agire

Finora abbiamo soltanto letto. Ora chiediamo al servizio di **cambiare** qualcosa, e un errore può avere conseguenze: spegnere il desktop sbagliato significa interrompere il lavoro di una persona.

Per questo adottiamo tre precauzioni, valide per qualunque automazione.

1. **Minimo privilegio.** Ogni utenza riceve solo i permessi indispensabili al proprio compito. Abbiamo letto con l'utenza `lettore`. Passiamo all'utenza `operatore` solo adesso, per il tempo necessario.
2. **Prima uno, poi tutti.** Proviamo l'azione su un solo desktop e controlliamo l'esito, prima di applicarla all'intero elenco.
3. **Prova a vuoto.** Prima di agire davvero facciamo dichiarare al programma che cosa farebbe, senza farlo. In inglese questa modalità si chiama *dry-run*.

### 9.1 Il lettore non può agire

Siamo ancora autenticati come `lettore`. Proviamo a spegnere un desktop e osserviamo la risposta.

In [ ]:
# Per le prove scegliamo il primo desktop dell'elenco: quello fermo da più tempo.
# .iloc[0] prende il primo valore della colonna.
nome_di_prova = da_spegnere["nome"].iloc[0]
print("Desktop scelto per la prova:", nome_di_prova)

# Richiesta POST all'endpoint che avvia lo spegnimento di quel desktop.
risposta = sessione.post(URL_BASE + "/desktop/" + nome_di_prova + "/spegni", timeout=ATTESA_MASSIMA)

print("Codice di stato:", risposta.status_code)
print("Messaggio      :", risposta.json()["errore"])

**Come leggere il risultato.** Il codice è `403`. Il servizio sa chi siamo, perché siamo autenticati. Il nostro ruolo, però, non comprende questa azione: non siamo autorizzati. È la differenza tra `401` e `403` vista nella sezione 5.

È una protezione preziosa. Se per errore avessimo lanciato lo spegnimento durante la fase di lettura, il servizio ci avrebbe fermati.

### 9.2 Cambiamo utenza

Ci autentichiamo come `operatore`, riusando la funzione `accedi`. Anche questa password, in un caso reale, arriverebbe da fuori del codice.

In [ ]:
ruolo = accedi("operatore", os.environ.get("VDI_PASSWORD_OPERATORE", "aula-operativa"))
print("Accesso eseguito con il ruolo:", ruolo)

### 9.3 Spegniamo un solo desktop, passo per passo

Spegnere una macchina richiede tempo. Il servizio non ci fa aspettare: risponde subito con il codice `202`, che significa «richiesta accettata, lavoro in corso», e ci consegna una ricevuta chiamata **operazione**, con un suo identificativo. Tocca poi a noi tornare a chiedere a che punto è.

Un'azione che prosegue dopo la risposta si dice **asincrona**. Funzionano così quasi tutte le azioni lente delle API reali: accendere, spegnere, creare, copiare.

Nella cella inviamo la richiesta, chiediamo subito lo stato dell'operazione, aspettiamo due secondi e lo chiediamo di nuovo.

In [ ]:
# 1) Chiediamo lo spegnimento.
risposta = sessione.post(URL_BASE + "/desktop/" + nome_di_prova + "/spegni", timeout=ATTESA_MASSIMA)
print("Codice di stato:", risposta.status_code)

operazione = risposta.json()
print("Operazione ricevuta:", operazione)

# 2) Chiediamo subito a che punto è l'operazione.
risposta = sessione.get(URL_BASE + "/operazioni/" + operazione["id"], timeout=ATTESA_MASSIMA)
print("Stato subito dopo      :", risposta.json()["stato"])

# 3) Aspettiamo due secondi e lo chiediamo di nuovo.
time.sleep(2)
risposta = sessione.get(URL_BASE + "/operazioni/" + operazione["id"], timeout=ATTESA_MASSIMA)
print("Stato dopo due secondi :", risposta.json()["stato"])

**Come leggere il risultato.** L'operazione è passata da `IN_CORSO` a `FATTA`. Verifichiamo sul servizio che il desktop sia davvero spento, leggendo i dati di quel solo desktop.

In [ ]:
risposta = sessione.get(URL_BASE + "/desktop/" + nome_di_prova, timeout=ATTESA_MASSIMA)
print(nome_di_prova, "->", risposta.json()["stato"])

### 9.4 Una funzione che attende la fine dell'operazione

Aspettare «due secondi» a occhio non è un metodo affidabile. Scriviamo una funzione che interroga il servizio a intervalli regolari, finché l'operazione non è conclusa. Questa tecnica si chiama *polling*.

La funzione ha un **tempo massimo**: se l'operazione non si conclude entro 30 secondi, la funzione smette di attendere e lo segnala. Un programma non deve mai aspettare all'infinito.

In [ ]:
def attendi_operazione(id_operazione, attesa_massima=30):
    """Interroga il servizio finché l'operazione indicata non è conclusa.

    Restituisce True se l'operazione si conclude entro "attesa_massima" secondi,
    False in caso contrario.
    """
    inizio = time.time()    # l'istante di partenza, in secondi

    # Continuiamo finché non è trascorso il tempo massimo.
    while time.time() - inizio < attesa_massima:
        risposta = sessione.get(URL_BASE + "/operazioni/" + id_operazione, timeout=ATTESA_MASSIMA)
        risposta.raise_for_status()

        if risposta.json()["stato"] == "FATTA":
            return True

        # Non è ancora finita: facciamo una breve pausa prima di chiedere di nuovo.
        time.sleep(0.5)

    return False


print("L'operazione di prima risulta conclusa:", attendi_operazione(operazione["id"]))

### 9.5 Una funzione che spegne, con la prova a vuoto

Raccogliamo tutto in una funzione. Il suo secondo argomento, `prova_a_vuoto`, vale `True` se non indichiamo altro. È una scelta voluta: **il comportamento predefinito è quello sicuro**. Per agire davvero dobbiamo scriverlo in modo esplicito.

La funzione restituisce un breve testo che descrive l'esito, così potremo raccogliere gli esiti in una tabella.

In [ ]:
def spegni_desktop(nome, prova_a_vuoto=True):
    """Chiede al servizio di spegnere il desktop indicato.

    Con prova_a_vuoto=True non invia nulla: dichiara soltanto che cosa farebbe.
    Restituisce un breve testo che descrive l'esito.
    """
    if prova_a_vuoto:
        return "prova a vuoto: nessuna richiesta inviata"

    risposta = sessione.post(URL_BASE + "/desktop/" + nome + "/spegni", timeout=ATTESA_MASSIMA)

    # Codice 403: la nostra utenza non ha il permesso. Lo riferiamo, senza interrompere il programma.
    if risposta.status_code == 403:
        return "rifiutata: permessi insufficienti"

    # Qualunque altro errore interrompe il programma.
    risposta.raise_for_status()

    # Codice 200: il desktop era già spento, il servizio non ha dovuto fare nulla.
    if risposta.status_code == 200:
        return "nessuna azione: era già spento"

    # Codice 202: richiesta accettata. Attendiamo la fine dell'operazione.
    operazione = risposta.json()
    if attendi_operazione(operazione["id"]):
        return "spento"
    return "operazione non conclusa nel tempo previsto"

### 9.6 Prova a vuoto su tutto l'elenco

Applichiamo la funzione a tutti i desktop dell'elenco `da_spegnere`, lasciando attiva la prova a vuoto. Raccogliamo gli esiti in una lista di dizionari, che poi trasformiamo in tabella.

In [ ]:
esiti = []

for nome in da_spegnere["nome"]:
    esito = spegni_desktop(nome)    # prova_a_vuoto vale True, perché non indichiamo altro
    esiti.append({"nome": nome, "esito": esito})

pd.DataFrame(esiti)

**Come leggere il risultato.** Nessuna richiesta è partita. Abbiamo però l'elenco esatto di ciò che accadrebbe. In un caso reale questo è il momento di fermarsi: l'elenco va fatto leggere a chi deve autorizzare l'intervento.

### 9.7 Ora davvero

Ripetiamo lo stesso ciclo con `prova_a_vuoto=False`. L'esecuzione richiede una decina di secondi, perché per ogni desktop attendiamo la fine dell'operazione.

In [ ]:
esiti = []

for nome in da_spegnere["nome"]:
    esito = spegni_desktop(nome, prova_a_vuoto=False)
    esiti.append({"nome": nome, "esito": esito})
    print(nome, "->", esito)

azioni = pd.DataFrame(esiti)

**Come leggere il risultato.** Il primo desktop risulta «già spento»: lo avevamo spento a mano nel paragrafo 9.3. Gli altri sette sono stati spenti ora.

Osserviamo che la funzione ha gestito da sola il caso del desktop già spento, senza errori. Ritroveremo questa proprietà alla fine della sezione 10.

### 9.8 Verifichiamo

Non ci fidiamo della nostra copia dei dati, che risale a prima dell'intervento. Rileggiamo l'inventario dal servizio e controlliamo lo stato degli 8 desktop.

In [ ]:
# Rileggiamo dal servizio l'inventario aggiornato.
inventario_dopo = pd.json_normalize(leggi_tutti_i_desktop())

# .isin(...) è vero per le righe il cui nome compare nell'elenco indicato.
controllo = inventario_dopo[inventario_dopo["nome"].isin(da_spegnere["nome"])]

controllo[["nome", "stato"]]

In [ ]:
# Il quadro generale dopo l'intervento.
inventario_dopo["stato"].value_counts()

**Come leggere il risultato.** Tutti gli 8 desktop sono ora nello stato `SPENTO`. Nel complesso i desktop accesi sono scesi da 43 a 35, e quelli spenti sono saliti da 5 a 13.

---
## 10. Passo 5 – Registrare

Un intervento su un'infrastruttura deve lasciare traccia: che cosa è stato fatto, quando, da chi, con quale esito. Le tracce sono due, e servono a scopi diversi.

### 10.1 La nostra traccia

Completiamo la tabella delle azioni con la data e con l'utenza usata, e la salviamo in un file.

In [ ]:
# Aggiungiamo data e ora dell'intervento, in un formato leggibile.
azioni["quando"] = pd.Timestamp.now().strftime("%d/%m/%Y %H:%M")

# Aggiungiamo l'utenza con cui abbiamo agito.
azioni["eseguito_da"] = "operatore"

azioni.to_csv("azioni_eseguite.csv", sep=";", index=False, encoding="utf-8-sig")

azioni

### 10.2 La traccia del servizio

Anche il servizio tiene un **registro** di tutte le azioni che gli sono state richieste. Nei sistemi reali si chiama spesso *audit log*, cioè «registro di controllo». Lo leggiamo dall'endpoint `/registro`.

In [ ]:
risposta = sessione.get(URL_BASE + "/registro", timeout=ATTESA_MASSIMA)
risposta.raise_for_status()

registro = pd.DataFrame(risposta.json()["elementi"])
registro

**Come leggere il risultato.** Il registro del servizio racconta tutta la nostra sessione. Tre osservazioni.

- Compare anche il **tentativo rifiutato**: lo spegnimento chiesto dall'utenza `lettore` nel paragrafo 9.1. I sistemi seri registrano anche ciò che non è riuscito.
- Ogni riga porta il nome di un'**utenza**. Per questo uno script non deve mai usare la password personale di un collega: tutte le sue azioni risulterebbero compiute da quella persona. Lo script deve avere un'utenza propria, creata apposta, con i soli permessi necessari. Un'utenza di questo tipo si chiama **utenza tecnica** oppure, nel linguaggio dei servizi cloud, **account di servizio**.
- Le due tracce non sono la stessa cosa. La nostra dice che cosa lo script **intendeva** fare. Quella del servizio dice che cosa è **realmente accaduto**, ed è quella che fa fede in caso di verifica.

### 10.3 Tutto insieme: dal notebook allo script

Abbiamo costruito il controllo un pezzo alla volta. Ora mettiamo i passi dal 2 al 5 in un'unica funzione, che riusa le funzioni già scritte. È la forma che il nostro lavoro avrebbe in uno **script**, cioè in un programma che si esegue per intero, senza fermarsi a ogni cella.

In [ ]:
def controllo_periodico(soglia_giorni=30, prova_a_vuoto=True):
    """Esegue l'intero controllo e restituisce la tabella delle azioni.

    Presuppone che l'accesso al servizio sia già stato eseguito (passo 1).
    """
    # Passo 2 - Leggere
    inventario = pd.json_normalize(leggi_tutti_i_desktop())
    inventario["ultimo_accesso"] = pd.to_datetime(inventario["ultimo_accesso"])
    inventario["giorni_inattivo"] = (pd.Timestamp.now() - inventario["ultimo_accesso"]).dt.days

    # Passo 3 - Decidere
    e_acceso = inventario["stato"] == "ACCESO"
    e_inutilizzato = inventario["giorni_inattivo"] > soglia_giorni
    candidati = inventario[e_acceso & e_inutilizzato]

    # Passo 4 - Agire (per finta, se non indichiamo altro)
    esiti = []
    for nome in candidati["nome"]:
        esito = spegni_desktop(nome, prova_a_vuoto=prova_a_vuoto)
        esiti.append({"nome": nome, "esito": esito})

    # Passo 5 - Registrare: restituiamo la tabella, che chi ci chiama potrà salvare
    return pd.DataFrame(esiti, columns=["nome", "esito"])


risultato = controllo_periodico()
print("Desktop da spegnere trovati:", len(risultato))

**Come leggere il risultato.** La funzione non trova più nulla da spegnere: i desktop inutilizzati sono già stati spenti poco fa.

È il comportamento giusto. Uno script ben fatto si può **rieseguire senza danni**: la seconda volta constata che il lavoro è già stato fatto e non fa nulla. Questa proprietà ha un nome, **idempotenza**. È ciò che permette di far partire uno script in automatico ogni settimana, senza timore.

> **Proviamo noi – Esercizio 5 (facoltativo).** Scriviamo la funzione `accendi_desktop(nome)`, sul modello di `spegni_desktop`, e usiamola per riaccendere `vdi-0005`. L'endpoint da usare è `/desktop/{nome}/accendi`. Poi rieseguiamo `controllo_periodico()`: che cosa trova?

<details>
<summary>Soluzione</summary>

```python
def accendi_desktop(nome):
    """Chiede al servizio di accendere il desktop indicato e attende la fine dell'operazione."""
    risposta = sessione.post(URL_BASE + "/desktop/" + nome + "/accendi", timeout=ATTESA_MASSIMA)
    risposta.raise_for_status()

    # Codice 200: il desktop era già acceso.
    if risposta.status_code == 200:
        return "nessuna azione: era già acceso"

    # Codice 202: attendiamo la fine dell'operazione.
    if attendi_operazione(risposta.json()["id"]):
        return "acceso"
    return "operazione non conclusa nel tempo previsto"


print(accendi_desktop("vdi-0005"))
print(controllo_periodico())
```

Dopo la riaccensione, `controllo_periodico()` trova di nuovo `vdi-0005`: è acceso, e l'ultimo accesso di un utente risale a molti mesi fa. Accendere la macchina non equivale a usarla.

</details>

In [ ]:
# Spazio per l'esercizio 5.

---
## 11. Finestra sul caso reale: Google Cloud

*Questa sezione è una dimostrazione: il codice si legge e si commenta, non si esegue in aula (assunzione A7). Riguarda da vicino chi gestisce i desktop virtuali. Per tutti gli altri è un esempio di come lo schema del laboratorio si ritrova, quasi identico, in un servizio reale.*

### 11.1 Di che cosa parliamo

Servono quattro definizioni.

- Il **cloud** è l'insieme di computer e servizi che un fornitore mette a disposizione attraverso la rete, e che si pagano in base all'uso. **Google Cloud** è l'offerta cloud di Google.
- **Compute Engine** è il servizio di Google Cloud che fornisce macchine virtuali. Nel suo vocabolario una macchina virtuale si chiama **istanza**.
- Un **progetto** è il contenitore in cui Google Cloud raggruppa le risorse di un cliente. Ogni progetto ha un proprio identificativo.
- Un **broker** è il programma che, in una VDI, assegna a ogni utente il suo desktop e gestisce la connessione.

Google non vende un prodotto VDI proprio. Quando parliamo di «Google VDI» intendiamo di solito questo: i desktop sono istanze di Compute Engine, e sopra di esse lavora un broker fornito da un altro produttore.

Esistono anche soluzioni di altro tipo, che portano all'utente singole applicazioni invece di desktop interi: un esempio è Cameyo by Google. A quelle questa sezione non si applica.

In questa sezione ci fermiamo allo strato di Compute Engine, che è lo stesso qualunque sia il broker. Tutto si regge sull'assunzione A6, che va verificata per prima cosa: **nella console di Google Cloud, i desktop compaiono nell'elenco delle istanze di Compute Engine?**

### 11.2 Dal simulatore a Google Cloud: la tabella di corrispondenza

Con il simulatore abbiamo scritto a mano ogni richiesta HTTP. Con Google Cloud non serve: Google pubblica una libreria Python, `google-cloud-compute`, che prepara le richieste al posto nostro. Una libreria di questo tipo, fornita dal produttore per usare la sua API da un linguaggio di programmazione, si chiama **SDK** (*Software Development Kit*).

Sotto la libreria ci sono richieste HTTP dello stesso genere di quelle che abbiamo imparato a scrivere. Per questo ogni elemento del laboratorio ha un corrispondente.

| Nel simulatore | Su Google Cloud | Nota |
|---|---|---|
| Richieste HTTP scritte a mano con `requests` | Libreria `google-cloud-compute` | La libreria costruisce e invia le richieste per noi |
| `POST /login`, che restituisce il token | *Application Default Credentials* | La libreria trova le credenziali nell'ambiente e ottiene il token da sola (paragrafo 11.3) |
| Utenze `lettore` e `operatore` | Ruoli **IAM** | IAM è la sigla di *Identity and Access Management*: il sistema che stabilisce chi può fare che cosa |
| `GET /desktop`, una pagina alla volta | Metodo `aggregated_list(...)` | La libreria scorre le pagine da sola |
| Stati `ACCESO` e `SPENTO` | Stati `RUNNING` e `TERMINATED` | Esistono anche stati di passaggio, per esempio `STOPPING` |
| Campo `etichette` | Campo `labels` | Coppie «chiave: valore» assegnate da chi gestisce le macchine |
| `POST /desktop/{nome}/spegni`, che restituisce un'operazione | Metodo `stop(...)`, che restituisce un oggetto operazione | L'esito si attende con il metodo `result()` |
| `GET /registro` | Cloud Audit Logs | L'arresto di un'istanza viene registrato tra le attività amministrative |
| Campo `ultimo_accesso` | **Non esiste in Compute Engine** | Vedi il paragrafo 11.6 |

### 11.3 Le credenziali

Nel codice che segue non compare alcuna password. La libreria usa un meccanismo chiamato **Application Default Credentials**, che possiamo tradurre con «credenziali predefinite dell'applicazione». Il meccanismo cerca le credenziali nell'ambiente in cui il programma gira, seguendo un ordine fisso.

1. Una variabile d'ambiente, `GOOGLE_APPLICATION_CREDENTIALS`, che indica un file di credenziali.
2. Le credenziali dell'utente, preparate una volta sul proprio PC con il comando `gcloud auth application-default login`. Il programma `gcloud` è lo strumento a riga di comando di Google Cloud.
3. L'account di servizio collegato alla macchina su cui gira il programma, quando il programma gira dentro Google Cloud.

È lo stesso principio del paragrafo 5.3, portato fino in fondo: **le credenziali vivono fuori dal codice**.

Due avvertenze.

- Un account di servizio può avere un **file di chiave**, cioè un file che contiene la sua credenziale permanente. Google stessa raccomanda di evitarlo quando esiste un'alternativa, perché un file si copia e si perde con facilità.
- Quale delle tre strade usare in banca non è una scelta di chi scrive lo script: spetta a chi governa la sicurezza.

### 11.4 La libreria

La libreria non fa parte dell'ambiente del corso. Chi vorrà provarla nel proprio ambiente la installerà con:

```text
python -m pip install google-cloud-compute
```

La cella seguente tenta di importarla. Usa il costrutto `try` ... `except`, che dice a Python: «esegui queste righe; se si verifica l'errore indicato, non fermarti ed esegui le righe alternative». Se la libreria non c'è, il notebook prosegue comunque: le funzioni dei paragrafi successivi vengono soltanto definite, non eseguite.

In [ ]:
# Proviamo a importare la libreria di Google. Se manca, ne prendiamo nota e proseguiamo.
try:
    from google.cloud import compute_v1
    LIBRERIA_GOOGLE_DISPONIBILE = True
except ImportError:
    LIBRERIA_GOOGLE_DISPONIBILE = False

print("Libreria google-cloud-compute disponibile:", LIBRERIA_GOOGLE_DISPONIBILE)

### 11.5 Leggere e agire

**Leggere.** La funzione seguente corrisponde alla nostra `leggi_tutti_i_desktop`. Restituisce una lista di dizionari con la stessa forma usata nel laboratorio, così il resto del lavoro (tabella, incrocio, report) resta identico.

Contiene una precauzione in più. Un progetto può ospitare macchine di ogni genere, non solo desktop. Per non toccare mai le macchine sbagliate, la funzione può tenere solo quelle che portano una certa **etichetta**, per esempio `ruolo = desktop`. Il nome dell'etichetta qui è inventato: quello vero va chiesto a chi gestisce le macchine.

In [ ]:
def elenca_macchine(id_progetto, etichetta_richiesta=None):
    """Legge da Compute Engine le macchine virtuali di un progetto.

    id_progetto: l'identificativo del progetto Google Cloud.
    etichetta_richiesta: una coppia (chiave, valore). Se è indicata, teniamo solo
        le macchine che portano quell'etichetta, per esempio ("ruolo", "desktop").

    Restituisce una lista di dizionari, uno per macchina.
    """
    # Il "client" è l'oggetto della libreria che parla con Compute Engine per nostro conto.
    client = compute_v1.InstancesClient()
    richiesta = compute_v1.AggregatedListInstancesRequest(project=id_progetto)

    macchine = []

    # aggregated_list restituisce le macchine raggruppate per zona.
    # La zona indica il centro di calcolo in cui la macchina si trova.
    # La libreria scorre da sola tutte le pagine dell'elenco.
    for zona, gruppo in client.aggregated_list(request=richiesta):
        for macchina in gruppo.instances:
            # dict(...) trasforma le etichette in un normale dizionario Python.
            etichette = dict(macchina.labels)

            # Se è richiesta un'etichetta e la macchina non la porta, la saltiamo.
            if etichetta_richiesta is not None:
                chiave, valore = etichetta_richiesta
                if etichette.get(chiave) != valore:
                    continue

            macchine.append({
                "nome": macchina.name,
                "stato": macchina.status,
                # La zona arriva nella forma "zones/europe-west8-a": teniamo l'ultima parte.
                "zona": zona.split("/")[-1],
                "ultimo_avvio": macchina.last_start_timestamp,
                "etichette": etichette,
            })

    return macchine

**Agire.** La funzione seguente corrisponde alla nostra `spegni_desktop`. Anche qui la prova a vuoto è il comportamento predefinito.

Il metodo `stop` restituisce subito un oggetto «operazione», come faceva il simulatore. Il metodo `result()` di quell'oggetto svolge il compito della nostra `attendi_operazione`: attende la conclusione, entro un tempo massimo.

In [ ]:
def ferma_macchina(id_progetto, zona, nome, prova_a_vuoto=True):
    """Chiede a Compute Engine di fermare una macchina virtuale.

    Con prova_a_vuoto=True non invia alcuna richiesta.
    Restituisce un breve testo che descrive l'esito.
    """
    if prova_a_vuoto:
        return "prova a vuoto: nessuna richiesta inviata"

    client = compute_v1.InstancesClient()

    # stop() avvia l'arresto e restituisce subito un oggetto "operazione".
    operazione = client.stop(project=id_progetto, zone=zona, instance=nome)

    # result() attende la conclusione dell'operazione, al massimo per 300 secondi.
    operazione.result(timeout=300)

    # Se l'operazione si è conclusa con un errore, lo riferiamo.
    if operazione.error_code:
        return "errore: " + str(operazione.error_message)

    return "fermata"

**Per provare davvero.** La cella seguente contatta Google Cloud solo se scriviamo l'identificativo di un progetto nella variabile `ID_PROGETTO`. Finché la variabile resta vuota, non parte alcuna richiesta. In ogni caso la cella si limita a leggere: non ferma nulla.

In [ ]:
# Identificativo di un progetto DI PROVA. Finché resta vuoto, la cella non contatta nessuno.
ID_PROGETTO = ""

if not LIBRERIA_GOOGLE_DISPONIBILE:
    print("Libreria di Google non installata: in aula ci limitiamo a leggere il codice.")
elif ID_PROGETTO == "":
    print("ID_PROGETTO è vuoto: nessuna richiesta inviata a Google Cloud.")
else:
    macchine = elenca_macchine(ID_PROGETTO, etichetta_richiesta=("ruolo", "desktop"))
    tabella_macchine = pd.json_normalize(macchine)
    print("Macchine trovate:", len(tabella_macchine))
    print(tabella_macchine.head())

### 11.6 Che cosa Compute Engine non sa

Nel laboratorio avevamo due informazioni decisive: l'**ultimo accesso** di un utente, fornito dal servizio, e l'**assegnazione**, fornita dal file. Compute Engine non conosce né l'una né l'altra.

- Compute Engine sa quando la macchina è stata **avviata** l'ultima volta (campo `last_start_timestamp`). Non sa quando un utente vi ha lavorato l'ultima volta: una macchina può restare accesa per mesi senza che nessuno la usi.
- Chi usa quale desktop, e quando, lo sanno altri sistemi: il **broker**, il sistema operativo del desktop, gli strumenti di monitoraggio, l'anagrafica aziendale delle utenze.

È il punto in cui il lavoro reale diventa specifico. Lo schema resta quello del laboratorio, ma la seconda fonte, quella da incrociare con l'inventario, va individuata caso per caso.

### 11.7 Prima di usarlo davvero

1. **Un progetto di prova.** Il codice di questa sezione non è stato eseguito su un progetto reale. La prima esecuzione va fatta dove un errore non fa danni.
2. **Un perimetro chiaro.** Lo script deve agire solo sulle macchine che gli competono. Le etichette servono a questo.
3. **Permessi minimi.** Per la sola lettura esiste un ruolo predefinito, Compute Viewer (`roles/compute.viewer`), che permette di elencare le risorse senza modificarle. Per fermare le macchine conviene un ruolo personalizzato che contenga solo i permessi necessari, per esempio `compute.instances.list` per elencare e `compute.instances.stop` per fermare. L'elenco esatto va definito e provato insieme a chi amministra il progetto.
4. **Attenzione al broker.** In una VDI è spesso il broker a governare accensione e spegnimento dei desktop. Prima di fermare una macchina «da sotto» va verificato come reagisce il broker: potrebbe riaccenderla, oppure segnalarla come guasta.
5. **Una modifica all'infrastruttura.** Uno script che ferma macchine va trattato come ogni altro intervento: autorizzazione, finestra di intervento, possibilità di tornare indietro.

---
## 12. Lo stesso schema, con il nostro sistema

I desktop virtuali erano un esempio. Ora ciascuno di noi applica lo schema al sistema di cui si occupa ogni giorno. Rispondiamo alle domande della tabella: la colonna centrale riporta, come traccia, le risposte del nostro laboratorio.

| Domanda | Nel laboratorio | Nel nostro sistema |
|---|---|---|
| Qual è il sistema? | Il servizio di gestione dei desktop virtuali | |
| Ha una API? Dove è descritta? | Sì. L'elenco degli endpoint ce lo ha dato il servizio stesso | |
| Come ci si autentica? | Nome utente e password, in cambio di un token | |
| Qual è l'inventario che possiamo leggere? | L'elenco dei desktop | |
| Con quale seconda fonte lo confrontiamo? | Il file delle assegnazioni | |
| Quale anomalia cerchiamo? | Desktop non assegnati; desktop accesi e inutilizzati | |
| Quale azione potremmo automatizzare? | Lo spegnimento | |
| Qual è il rischio di quell'azione? | Spegnere un desktop che qualcuno sta usando | |
| Chi deve autorizzarla? | Il responsabile dei desktop virtuali | |

**Come scoprire se un sistema ha una API.** Nella documentazione del prodotto cerchiamo le parole «API», «REST», «SDK», «automation» oppure «developer». Se il sistema ha una console web, quasi sempre ha anche una API: spesso la console stessa la usa dietro le quinte.

**Un consiglio per cominciare.** Conviene che il primo script si fermi al passo 3: leggere, confrontare, produrre un report. Non modifica nulla, quindi è facile da far autorizzare, e mostra subito il valore del lavoro. L'azione automatica può arrivare in un secondo momento.

---
## 13. Quando Python conviene, e quando no

Python non è sempre lo strumento giusto. Ecco qualche criterio per decidere.

**Conviene quando:**

- c'è una **logica** da applicare: confronti, soglie, eccezioni;
- ci sono **più fonti** da incrociare, come un servizio e un file;
- il controllo va **ripetuto** nel tempo, sempre allo stesso modo;
- serve un **report**, oppure una traccia di ciò che è stato fatto.

**Non conviene quando:**

- l'operazione è **singola e occasionale**: si fa prima dalla console;
- il sistema offre già uno **strumento a riga di comando** che fa esattamente ciò che serve con un solo comando;
- il sistema ha già una **funzione integrata** per quel compito, per esempio una pianificazione di accensioni e spegnimenti. Una funzione integrata è mantenuta dal produttore, uno script no;
- manca un **ambiente di prova** in cui collaudare lo script senza rischi.

Chi lavora già con un altro linguaggio di scripting non deve abbandonarlo. Python si affianca agli strumenti esistenti, e dà il meglio quando c'è da ragionare sui dati prima di agire.

---
## 14. Riepilogo

**I cinque passi, e ciò che abbiamo usato per ciascuno.**

| Passo | Che cosa abbiamo fatto | Strumenti |
|---|---|---|
| 1. Autenticarsi | Abbiamo ottenuto un token e lo abbiamo memorizzato nella sessione | `sessione.post`, intestazione `Authorization` |
| 2. Leggere | Abbiamo scaricato l'inventario, tutte le pagine | `sessione.get`, `params=`, ciclo `while` |
| 3. Decidere | Abbiamo incrociato inventario e assegnazioni e trovato le anomalie | `pd.json_normalize`, `merge`, maschere |
| 4. Agire | Abbiamo spento i desktop inutilizzati, prima per finta e poi davvero | `prova_a_vuoto`, attesa dell'operazione |
| 5. Registrare | Abbiamo salvato report e azioni e letto il registro del servizio | `to_csv`, endpoint `/registro` |

**Le abitudini da portare con noi.**

1. Le credenziali vivono **fuori dal codice**.
2. Guardiamo sempre il **codice di stato**, e ci fermiamo subito se segnala un errore.
3. Leggiamo **tutte le pagine**, e confrontiamo il numero di elementi letti con il totale dichiarato.
4. Leggiamo con un'utenza di sola lettura. Passiamo a un'utenza con più permessi **solo quando serve**.
5. Prima di agire facciamo una **prova a vuoto**, e proviamo su **un solo elemento**.
6. Ogni attesa ha un **tempo massimo**.
7. Dopo aver agito **rileggiamo** dal servizio, e lasciamo una **traccia**.
8. Scriviamo script che si possano **rieseguire senza danni**.

**I file creati nella cartella del notebook.**

| File | Contenuto |
|---|---|
| `assegnazioni.csv` | Il file delle assegnazioni, scritto dal simulatore all'avvio |
| `report_desktop_da_spegnere.csv` | L'elenco dei desktop accesi e inutilizzati, con il costo mensile |
| `azioni_eseguite.csv` | L'elenco delle azioni eseguite, con l'esito di ciascuna |

---
## Glossario

| Termine | Significato |
|---|---|
| **Account di servizio** | Utenza creata per un programma e non per una persona. È detta anche *utenza tecnica* |
| **API** | *Application Programming Interface*. L'elenco delle operazioni che un sistema mette a disposizione degli altri programmi, con le regole per richiederle |
| **Asincrona (azione)** | Azione che prosegue dopo la risposta del servizio. Il servizio consegna subito una ricevuta, l'operazione, e il lavoro continua in sottofondo |
| **Autenticazione** | Il controllo dell'identità: «chi siamo?» |
| **Autorizzazione** | Il controllo dei permessi: «che cosa ci è permesso fare?» |
| **Broker** | In una VDI, il programma che assegna a ogni utente il suo desktop e gestisce la connessione |
| **Chiave** | La colonna che collega due tabelle e permette di incrociarle |
| **Cloud** | Computer e servizi messi a disposizione da un fornitore attraverso la rete, e pagati in base all'uso |
| **Codice di stato** | Il numero con cui ogni risposta HTTP dice com'è andata la richiesta, per esempio `200` oppure `401` |
| **Console** | La pagina web o il programma con cui si amministra un sistema a mano |
| **CSV** | *Comma-Separated Values*. File di testo che contiene una tabella, con le colonne divise da un separatore |
| **DataFrame** | La tabella di pandas |
| **Dati annidati** | Dati in cui un dizionario ne contiene un altro |
| **Desktop virtuale** | Computer da ufficio che risiede in un centro di calcolo e si usa attraverso la rete |
| **Docstring** | Il testo tra triple virgolette che descrive che cosa fa una funzione |
| **Endpoint** | Un indirizzo a cui il servizio risponde per una certa operazione |
| **Etichetta** | Coppia «chiave: valore» che chi gestisce le macchine assegna a ciascuna per classificarla |
| **Gestore dei segreti** | Archivio protetto in cui un'organizzazione conserva le password e le altre credenziali dei programmi |
| **GET** | Il tipo di richiesta HTTP che serve a leggere |
| **HTTP** | Il protocollo con cui viaggiano le pagine web e la maggior parte delle API |
| **IAM** | *Identity and Access Management*. In Google Cloud, il sistema che stabilisce chi può fare che cosa |
| **Idempotenza** | La proprietà di un'azione che si può ripetere senza produrre effetti ulteriori |
| **Infrastruttura** | L'insieme delle risorse tecniche che fanno funzionare i servizi informatici |
| **Intestazione** | Riga di informazioni di servizio che accompagna una richiesta o una risposta HTTP. In inglese *header* |
| **Inventario** | L'elenco delle risorse gestite, con le loro caratteristiche |
| **ISO 8601** | Il formato internazionale per scrivere date e ore, per esempio `2026-11-17T09:30:00` |
| **Istanza** | Il nome che Compute Engine dà a una macchina virtuale |
| **JSON** | *JavaScript Object Notation*. Il formato di testo più usato per scambiare dati tra programmi |
| **Kernel** | Il programma Python che esegue le celle del notebook |
| **Macchina virtuale** | Computer realizzato via software, che funziona insieme ad altri sullo stesso computer fisico |
| **Maschera** | Colonna di valori «vero» o «falso», usata per selezionare le righe di una tabella |
| **Metodo** | In Python, una funzione che appartiene a un oggetto e si richiama con un punto, come in `risposta.json()` |
| **Minimo privilegio** | Il principio per cui ogni utenza riceve solo i permessi indispensabili al proprio compito |
| **NaN** | *Not a Number*. Il modo in cui pandas indica un valore mancante |
| **Operazione** | La ricevuta, con un identificativo, che il servizio consegna quando avvia un'azione asincrona |
| **Pacchetto** | Raccolta di codice già scritto, che si aggiunge a Python. È detto anche *libreria* |
| **Paginazione** | La consegna di un elenco lungo a blocchi, chiamati pagine, uno per richiesta |
| **Parametri della richiesta** | Valori che si aggiungono a una richiesta per precisarla, per esempio il numero della pagina voluta |
| **Polling** | Interrogare un servizio a intervalli regolari, in attesa che qualcosa cambi |
| **Porta** | Il numero che distingue i diversi servizi in ascolto sullo stesso computer |
| **POST** | Il tipo di richiesta HTTP che serve a inviare dati o a chiedere un'azione |
| **Progetto** | In Google Cloud, il contenitore che raggruppa le risorse di un cliente |
| **Protocollo** | Insieme di regole che due programmi seguono per capirsi |
| **Prova a vuoto** | Esecuzione in cui il programma dichiara che cosa farebbe, senza farlo. In inglese *dry-run* |
| **Proxy** | Computer intermedio da cui passano, in una rete aziendale, le richieste dirette a Internet |
| **Registro** | La traccia delle azioni richieste a un servizio. In inglese *audit log* |
| **REST** | Lo stile più diffuso di API: richieste HTTP a indirizzi ben definiti, risposte di solito in JSON |
| **Richiesta e risposta** | I due messaggi di ogni scambio HTTP: il programma invia una richiesta, il servizio restituisce una risposta |
| **Riconciliazione** | Il confronto tra due elenchi che dovrebbero descrivere la stessa realtà, per trovare le differenze |
| **Ruolo** | L'insieme dei permessi assegnati a un'utenza |
| **Script** | Programma che si esegue per intero, dall'inizio alla fine |
| **SDK** | *Software Development Kit*. Libreria fornita da un produttore per usare la sua API da un linguaggio di programmazione |
| **Servizio web** | Programma che resta in ascolto, riceve richieste HTTP e risponde |
| **Sessione** | In `requests`, l'oggetto che ricorda le impostazioni comuni a tutte le richieste |
| **Simulatore** | Programma che imita il comportamento di un sistema reale, per esercitarsi senza rischi |
| **Token** | Sequenza casuale di caratteri che vale come lasciapassare temporaneo al posto della password |
| **Variabile d'ambiente** | Valore conservato dal sistema operativo, che un programma può leggere |
| **VDI** | *Virtual Desktop Infrastructure*. L'insieme delle tecnologie che rendono possibili i desktop virtuali |
| **Zona** | In Google Cloud, il centro di calcolo in cui si trova una risorsa, per esempio `europe-west8-a` a Milano |

---
## Appendice A. Come è fatto il simulatore (facoltativa)

La cella di servizio del paragrafo 3.2 è divisa in quattro blocchi, segnalati da commenti numerati.

| Blocco | Che cosa contiene |
|---|---|
| 1. I dati inventati | Le utenze finte e le funzioni che creano i 48 desktop e il file delle assegnazioni. I dati nascono da un generatore di numeri casuali avviato con un «seme» fisso, cioè con un numero di partenza che gli fa produrre sempre la stessa sequenza: per questo sono identici su ogni PC |
| 2. La memoria del servizio | Un oggetto che conserva i desktop, i token validi, le operazioni avviate e il registro delle azioni |
| 3. Lo sportello | La parte che riceve le richieste HTTP. Il metodo `do_GET` risponde alle richieste di lettura, il metodo `do_POST` a quelle che inviano dati o chiedono un'azione. Ogni risposta è composta da un codice di stato e da un testo JSON |
| 4. Accensione e spegnimento | Le funzioni `avvia_simulatore()` e `ferma_simulatore()` |

Alcune cose utili da sapere.

- Il simulatore è costruito con il modulo `http.server`, che fa parte di Python: non richiede installazioni.
- Gira **in sottofondo**, dentro il kernel, cioè dentro lo stesso programma Python che esegue le celle del notebook. Per questo possiamo continuare a eseguire celle mentre il servizio resta in ascolto.
- Si spegne da solo quando chiudiamo il notebook o riavviamo il kernel. Per spegnerlo a mano basta eseguire `ferma_simulatore()` in una cella.
- Leggere il blocco 3 è un buon modo per capire che cosa accade **dall'altra parte dello sportello**: come un servizio controlla il token, verifica i permessi e sceglie il codice di stato.

---
## Appendice B. Se qualcosa non funziona

| Che cosa vediamo | Causa probabile | Rimedio |
|---|---|---|
| `ModuleNotFoundError` nella sezione 2 | Manca il pacchetto `requests` oppure `pandas` | Installare il pacchetto come indicato nella sezione 2 e riavviare il notebook |
| `NameError: name '...' is not defined` | Una cella precedente non è stata eseguita | Rieseguire le celle dall'inizio, in ordine |
| `OSError` oppure `PermissionError` all'avvio del simulatore (paragrafo 3.3) | Il PC non consente di aprire un servizio locale, oppure la cartella del notebook non è scrivibile | Spostare il notebook in una cartella in cui possiamo scrivere. Se l'errore resta, segnalarlo al docente: va rivista la configurazione del PC |
| `ConnectionError` oppure «connessione rifiutata» | Il simulatore è spento, per esempio dopo un riavvio del kernel | Rieseguire le celle dei paragrafi 3.2 e 3.3, poi quelle delle sezioni 4 e 5 |
| Codice `401` dopo un riavvio del simulatore | Il token ottenuto prima del riavvio non vale più | Rieseguire `accedi(utente, password)` |
| `PermissionError` durante il salvataggio di un file CSV | Il file è aperto in Excel, oppure la cartella non è scrivibile | Chiudere il file in Excel, oppure spostare il notebook in una cartella in cui possiamo scrivere |
| I numeri non coincidono con quelli del testo | Il laboratorio è stato eseguito in parte, e alcuni desktop sono già stati spenti | Rieseguire la cella del paragrafo 3.3, che riporta il simulatore ai dati iniziali, e ripartire dalla sezione 5 |